In [ ]:
# --- SPACEMAP notebook configuration ------------------------------------
# Every path below is built from DATA_DIR. Point it at your own data by
# setting SPACEMAP_DATA_DIR in your environment, or by editing the default.
import os
from pathlib import Path

DATA_DIR = os.environ.get("SPACEMAP_DATA_DIR", str(Path.home() / "spacemap-data"))
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Using DATA_DIR = {DATA_DIR}")


### Phenotype comparison Analysis

This notebook provides a comparative analysis of the four phenotyping methods: Leiden, SOM, Threshold-based, and SCIMAP. It takes the unified phenotyping file as input and generates several visualizations to compare the results.

## Data Quality: Fix Typos and Inconsistencies

This cell corrects known data quality issues in the unified CSV file:
- Fixes typo in 'phenotype_leiden': 'unlcassified' → 'unclassified'
- Preserves all other data unchanged
- Only modifies the CSV on disk if changes are made

In [ ]:
import pandas as pd
from pathlib import Path

# Define the path to the unified CSV
project_root = Path().resolve().parent
input_csv = project_root / "outputs/phenotyping" / "all_samples_combined_corrected_phenotyped.csv"

# Load the CSV
print(f"Loading CSV: {input_csv}")
df = pd.read_csv(input_csv)

# Check if 'phenotype_leiden' column exists
if 'phenotype_leiden' in df.columns:
    # Count typos before fix
    typo_count_before = (df['phenotype_leiden'] == 'unlcassified').sum()
    
    if typo_count_before > 0:
        print(f"\nFound {typo_count_before} instances of 'unlcassified' in 'phenotype_leiden' column")
        
        # Fix the typo
        df['phenotype_leiden'] = df['phenotype_leiden'].replace('unlcassified', 'unclassified')
        
        # Verify the fix
        typo_count_after = (df['phenotype_leiden'] == 'unlcassified').sum()
        correct_count = (df['phenotype_leiden'] == 'unclassified').sum()
        
        print(f"After fix:")
        print(f"  Remaining 'unlcassified': {typo_count_after}")
        print(f"  Total 'unclassified': {correct_count}")
        
        # Save the corrected CSV back (overwrite)
        df.to_csv(input_csv, index=False)
        print(f"\n✓ CSV updated and saved: {input_csv}")
    else:
        print("\nNo typos found. CSV is already clean.")
else:
    print("\nWarning: 'phenotype_leiden' column not found in CSV")

print("\nData quality check complete!")

# Step 1: Load Data & Generate Phenotype Proportion Summary

This step:
- Loads the unified phenotyping CSV (with all methods appended).
- Defines the phenotype columns to compare (`phenotype_leiden`, `phenotype_som`, `phenotype_threshold`, `phenotype_scimap`).
- Computes per‑method phenotype counts and proportions (long format table).
- Derives a consistent phenotype ordering by mean proportion across methods.
- Saves:
  - `phenotype_proportions_long.csv` (long tidy table: method, phenotype, count, total_cells, proportion)
  - `phenotype_proportions_pivot.csv` (phenotype × method proportion matrix)
- Produces and saves a grouped bar plot (`phenotype_proportions_grouped.png`) comparing phenotype prevalence across methods.

Notes:
- Rows with NaN phenotype labels are skipped.
- Proportions are per method (count / total cells for that method column).
- The plot uses a unified order of phenotypes for easier visual comparison.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import os

project_root = Path().resolve().parent
#INPUT_CSV = project_root / "outputs/phenotyping" / "all_samples_combined_corrected_phenotyped.csv"
INPUT_CSV = Path(f'{DATA_DIR}/example_study/registration/roi/cellpose/annotated_csv/corrected/batch_corrected_merged.csv')
OUTPUT_DIR = project_root / "outputs/phenotyping" / "comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV)

columns_to_process = [
    'phenotype_leiden',
    'phenotype_som',
    'phenotype_threshold',
    'phenotype_scimap'
]

# Build long summary table
records = []
for method in columns_to_process:
    vc = df[method].value_counts(dropna=False)
    total = int(vc.sum())
    for phenotype, count in vc.items():
        if pd.isna(phenotype):
            continue
        proportion = count / total
        records.append({
            'method': method,
            'phenotype': str(phenotype),
            'count': int(count),
            'total_cells': total,
            'proportion': proportion
        })

summary_long = pd.DataFrame(records)

# Order phenotypes by mean proportion (descending)
phenotype_order = (summary_long.groupby('phenotype')['proportion']
                   .mean().sort_values(ascending=False).index.tolist())

# Save tables
long_path = OUTPUT_DIR / 'phenotype_proportions_long.csv'
summary_long.to_csv(long_path, index=False)
pivot_prop = summary_long.pivot(index='phenotype', columns='method', values='proportion').fillna(0)
pivot_prop_path = OUTPUT_DIR / 'phenotype_proportions_pivot.csv'
pivot_prop.to_csv(pivot_prop_path)

print(f"Saved long format table to: {long_path}")
print(f"Saved pivot proportions table to: {pivot_prop_path}")
print('\nLong format head:')
print(summary_long.head())

# Grouped bar plot
palette = sns.color_palette('tab10', n_colors=len(columns_to_process))
plt.figure(figsize=(22, 8))
sns.barplot(data=summary_long, x='phenotype', y='proportion', hue='method',
            order=phenotype_order, palette=palette)
plt.title('Phenotype Proportions Across Methods')
plt.ylabel('Proportion of Cells')
plt.xlabel('Phenotype')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Method', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
grouped_fig_path = OUTPUT_DIR / 'phenotype_proportions_grouped.png'
plt.savefig(grouped_fig_path, dpi=300)
print(f"Grouped proportion figure saved to: {grouped_fig_path}")
plt.show()

## 2. Pairwise Phenotype Agreement (Consistency Matrix)

This heatmap shows the percentage of cells that are assigned the same phenotype between each pair of methods. This quantifies the overall agreement between any two phenotyping strategies.

In [ ]:


#%% Pairwise comparison of phenotping methods
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import itertools
from pathlib import Path
#from spacemap.config import default_config

project_root = Path().resolve().parent


# Unified phenotyping output file
#INPUT_CSV = project_root / "outputs/phenotyping" / "all_samples_combined_corrected_phenotyped.csv"
#INPUT_CSV = project_root / "outputs/phenotyping/batch/circles" / "P29_P30_roi_2_1_circle2.csv"
INPUT_CSV = f'{DATA_DIR}/example_study/registration/roi/cellpose/annotated_csv/corrected/batch_corrected_merged.csv'

OUTPUT_DIR = project_root / "outputs/phenotyping" / "comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output will be saved to: {OUTPUT_DIR}")

# The phenotype columns you want to compare
METHODS_TO_COMPARE = [
    'phenotype_leiden',
    'phenotype_som',
    'phenotype_threshold',
    'phenotype_scimap',
    #'phenotype_ref1',
    #'phenotype_ref2',
]

#METHODS_TO_COMPARE = ['phenotype_leiden','phenotype_som','phenotype_threshold','phenotype_scimap','phenotype_ref1','phenotype_ml_consensus','phenotype_consensus_refined']
# -------------------------------------------------------------------------------

print("Starting pairwise comparison of phenotyping methods...")

# 1) Create the output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output will be saved to: {OUTPUT_DIR}")

# 2) Load the data
try:
    df = pd.read_csv(INPUT_CSV)
    # Keep only the columns we need for the comparison
    df = df[METHODS_TO_COMPARE]
    print(f"Successfully loaded data from {os.path.basename(INPUT_CSV)}")
except FileNotFoundError:
    print(f"ERROR: Input file not found at {INPUT_CSV}")
    exit()
except KeyError:
    print(f"ERROR: One or more columns from {METHODS_TO_COMPARE} not found in the input file.")
    exit()

# 3) Initialize a DataFrame to store the similarity matrix
similarity_matrix = pd.DataFrame(
    np.zeros((len(METHODS_TO_COMPARE), len(METHODS_TO_COMPARE))),
    index=METHODS_TO_COMPARE,
    columns=METHODS_TO_COMPARE
)

# 4) Calculate pairwise similarity
print("\nCalculating similarity matrix...")

# Use itertools to efficiently get all unique pairs of methods
for method1, method2 in itertools.combinations_with_replacement(METHODS_TO_COMPARE, 2):
    print(f"  Comparing: {method1} vs {method2}")

    # Create a temporary dataframe for the pair and drop rows with nulls
    filtered_df = df[[method1, method2]].dropna()
    
    # *** FIX: Handle potential duplicate columns by ensuring we always get a Series. ***
    s1 = filtered_df[method1]
    if isinstance(s1, pd.DataFrame):
        print(f"    Warning: Duplicate columns found for '{method1}'. Using the first instance.")
        s1 = s1.iloc[:, 0]

    s2 = filtered_df[method2]
    if isinstance(s2, pd.DataFrame):
        print(f"    Warning: Duplicate columns found for '{method2}'. Using the first instance.")
        s2 = s2.iloc[:, 0]

    if s1.empty:
        similarity = 0.0
    else:
        # Now s1 and s2 are guaranteed to be Series
        matches = (s1 == s2).sum()
        total_comparisons = len(s1)
        similarity = float((matches / total_comparisons) * 100)

    # Use .at for setting a single scalar value.
    similarity_matrix.at[method1, method2] = similarity
    similarity_matrix.at[method2, method1] = similarity

# 5) Save the similarity matrix to a CSV file
matrix_csv_path = os.path.join(OUTPUT_DIR, 'pairwise_similarity_matrix.csv')
similarity_matrix.to_csv(matrix_csv_path)
print(f"\nSimilarity matrix saved to: {matrix_csv_path}")
print("\nSimilarity Matrix (% Agreement):")
print(similarity_matrix.round(1))

# 6) Generate and save the heatmap visualization
print("\nGenerating heatmap...")

# Create a mask for the upper triangle to make it a triangular heatmap
mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)

plt.figure(figsize=(10, 8))
heatmap = sns.heatmap(
    similarity_matrix,
    mask=mask,  # This creates the triangular effect
    annot=True,
    fmt='.1f',
    annot_kws={'size': 20},  # Increase font size of the numbers in cells
    cmap='YlGnBu',
    linewidths=.5,
    vmin=0,
    vmax=100,
    cbar_kws={'label': 'Similarity Percentage (%)'}
)

plt.title('Pairwise Method Similarity (% of matching phenotypes)', fontsize=16, pad=20)
plt.xticks(rotation=90, ha='right', fontsize=12)  # Increase axis label font size
plt.yticks(rotation=0, fontsize=12)  # Increase axis label font size
heatmap.set_xlabel('')
heatmap.set_ylabel('')

plt.tight_layout()
heatmap_path = os.path.join(OUTPUT_DIR, 'pairwise_similarity_heatmap.png')
plt.savefig(heatmap_path, dpi=300)
plt.show() # This will display the figure in the notebook
plt.close() # This will close the figure object to free up memory

print(f"Heatmap saved as: {heatmap_path}")
print("\nProcess complete.")
# %%


In [ ]:
#F1 score

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from collections import Counter
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
# Use the script's location to define a robust project root path
project_root = Path.cwd().parent

#INPUT_FILE = project_root / 'outputs/phenotyping' / 'all_samples_combined_corrected_phenotyped_C4.csv'
INPUT_FILE = f'{DATA_DIR}/example_study/registration/roi/cellpose/annotated_csv/corrected/batch_corrected_merged.csv'
#INPUT_FILE = f'{DATA_DIR}/example_project/outputs/phenotyping/all_samples_combined_corrected_phenotyped_final_v2.csv'
#INPUT_FILE = f'{DATA_DIR}/example_project/outputs/phenotyping/all_samples_combined_corrected_phenotyped.csv'

OUTPUT_DIR = project_root / 'outputs/phenotyping/comparison_ref1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

GROUND_TRUTH_COLUMN = 'phenotype_ref1'
DATASET_ID_COLUMN = 'batch_id'

# Phenotyping methods to analyze
PHENOTYPING_METHODS = [
    'phenotype_leiden',
    'phenotype_som',
    'phenotype_threshold',
    'phenotype_scimap',
    'phenotype_ml_mr',
    'phenotype_ml_consensus',
    #'phenotype_consensus_refined',
]

# Consensus analysis options
INCLUDE_CONSENSUS_ANALYSIS = True  # Set to False to skip consensus entirely
CONSENSUS_SEPARATE_ANALYSIS = False  # Set to True to run consensus analysis separately (only if INCLUDE_CONSENSUS_ANALYSIS is True)

# Non-consensus analysis options
INCLUDE_NONCONSENSUS_ANALYSIS = False  # Set to False to skip non-consensus analysis
NONCONSENSUS_PHENOTYPE_COLUMN = 'phenotype_consensus_refined'  # Column to use for non-consensus cells
NONCONSENSUS_SCORE_RANGE = (0.75, 1.0)  # ML consensus score range filter (min, max)
NONCONSENSUS_SEPARATE_ANALYSIS = False  # Set to True to run non-consensus analysis separately

# Consensus requirements
MIN_CONSENSUS_METHODS = 4

# Filter configuration for visualization
# Phenotypes to exclude from comprehensive_per_class_analysis.png (e.g., low-frequency phenotypes)
FILTER_OUT_PHENOTYPES = [
    #'DC',           # Low frequency phenotype
    #'Lymph',        # Low frequency phenotype  
    #'Muscle',
    #'Nerve',
    #'Stroma',
    'unclassified', # Uninformative category
    'unknown',      # Uninformative category
    'Others',       # Uninformative category
    # Add more phenotypes to filter out as needed
]
# This helps focus on well-represented cell types in the comprehensive_per_class_analysis.png

# Figure settings
plt.style.use('default')
sns.set_palette("husl")
FIGURE_DPI = 300


# =============================================================================
# DATA PREPARATION
# =============================================================================
def prepare_data_for_analysis(input_file):
    """
    Loads and prepares the main CSV for validation analysis.
    """
    print(f"Preparing data from: {input_file}")
    try:
        df = pd.read_csv(input_file)

        if GROUND_TRUTH_COLUMN not in df.columns:
            print(f"ERROR: Ground truth column '{GROUND_TRUTH_COLUMN}' not found.")
            return None
        df.rename(columns={GROUND_TRUTH_COLUMN: 'ground_truth'}, inplace=True)

        if DATASET_ID_COLUMN not in df.columns:
            print(f"ERROR: Dataset ID column '{DATASET_ID_COLUMN}' not found.")
            return None
        df.rename(columns={DATASET_ID_COLUMN: 'dataset_name'}, inplace=True)

        # Filter out rows with invalid ground truth labels
        bad_labels = ['unclassified', ]
        df = df[~df['ground_truth'].isin(bad_labels)].copy()
        df.dropna(subset=['ground_truth'], inplace=True)

        print("Data preparation successful.")
        print(f"  Total cells for analysis: {len(df)}")
        print(f"  Ground truth cell types: {df['ground_truth'].nunique()}")
        print(f"  Datasets found: {df['dataset_name'].nunique()}")
        return df

    except FileNotFoundError:
        print(f"ERROR: Input file not found: {input_file}")
        return None
    except Exception as e:
        print(f"ERROR during data preparation: {e}")
        return None


# =============================================================================
# INDIVIDUAL METHOD VALIDATION
# =============================================================================
def validate_individual_methods(df, methods):
    """Validate each individual phenotyping method against ground truth."""
    print("Validating individual phenotyping methods...")
    individual_results = []
    available_methods = [m for m in methods if m in df.columns]

    for method in available_methods:
        print(f"  Analyzing {method}...")

        method_df = df[
            df[method].notna() & (df[method] != 'unknown') & df['ground_truth'].notna()
        ].copy()

        if len(method_df) == 0:
            print(f"    No valid predictions found for {method}")
            continue

        y_true = method_df['ground_truth']
        y_pred = method_df[method]

        labels = sorted(list(set(y_true) | set(y_pred)))
        precision_per, recall_per, f1_per, support_per = precision_recall_fscore_support(
            y_true, y_pred, labels=labels, average=None, zero_division=0
        )
        precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0
        )

        results = {
            'method': method,
            'accuracy': accuracy_score(y_true, y_pred),
            'precision_weighted': precision_w,
            'recall_weighted': recall_w,
            'f1_weighted': f1_w,
            'cohen_kappa': cohen_kappa_score(y_true, y_pred),
            'matthews_corr': matthews_corrcoef(y_true, y_pred),
            'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels),
            'labels': labels,
            'per_class_precision': precision_per,
            'per_class_recall': recall_per,
            'per_class_f1': f1_per,
            'per_class_support': support_per,
            'n_cells': len(method_df),
            'coverage': (len(method_df) / len(df)) * 100
        }
        individual_results.append(results)
        print(f"    {method}: {results['accuracy']:.3f} accuracy ({len(method_df)} cells)")

    return individual_results


# =============================================================================
# CONSENSUS ANALYSIS (PERFORMANCE OPTIMIZED)
# =============================================================================
def identify_consensus_cells(df, methods, min_agreement):
    """
    Identify consensus cells where at least min_agreement methods agree.
    (Vectorized for high performance)
    """
    print(f"Identifying consensus cells (>={min_agreement} methods agreeing)...")
    
    df_consensus = df.copy()
    available_methods = [m for m in methods if m in df.columns]
    print(f"  Available methods: {available_methods}")

    preds_df = df_consensus[available_methods]

    valid_predictions_count = preds_df.replace('unknown', np.nan).notna().sum(axis=1)
    consensus_phenotype = preds_df.mode(axis=1)[0]
    consensus_count = preds_df.eq(consensus_phenotype, axis=0).sum(axis=1)
    is_consensus = (consensus_count >= min_agreement) & (valid_predictions_count >= min_agreement)

    df_consensus['consensus_phenotype'] = consensus_phenotype
    df_consensus['consensus_count'] = consensus_count
    df_consensus['is_consensus'] = is_consensus
    
    df_consensus.loc[~is_consensus, ['consensus_phenotype', 'consensus_count']] = (None, 0)
    
    def get_agreeing_methods(row):
        if not row['is_consensus']:
            return ''
        return ', '.join([m for m in available_methods if row[m] == row['consensus_phenotype']])
        
    df_consensus['consensus_methods'] = df_consensus.apply(get_agreeing_methods, axis=1)

    consensus_rate = (df_consensus['is_consensus'].sum() / len(df_consensus)) * 100
    print(f"  Found {df_consensus['is_consensus'].sum()} consensus cells ({consensus_rate:.1f}%)")
    
    return df_consensus


def identify_nonconsensus_cells(df_consensus, nonconsensus_col, score_range):
    """
    Identify non-consensus cells and apply ML consensus score filtering.
    """
    print(f"Identifying non-consensus cells using '{nonconsensus_col}' column...")
    
    # Create non-consensus mask
    df_nonconsensus = df_consensus.copy()
    is_nonconsensus = ~df_consensus['is_consensus']
    
    # Apply ML consensus score filtering if the score column exists
    score_col = 'ml_consensus_score'
    if score_col in df_nonconsensus.columns and score_range:
        min_score, max_score = score_range
        print(f"  Applying ML consensus score filter: {min_score} <= score <= {max_score}")
        
        score_mask = (df_nonconsensus[score_col] >= min_score) & (df_nonconsensus[score_col] <= max_score)
        is_nonconsensus = is_nonconsensus & score_mask
        
        filtered_count = score_mask.sum()
        total_nonconsensus = (~df_consensus['is_consensus']).sum()
        print(f"  Score filter applied: {filtered_count}/{total_nonconsensus} non-consensus cells meet score criteria")
    else:
        if score_range:
            print(f"  WARNING: '{score_col}' column not found, skipping score filtering")
    
    # Add non-consensus information
    df_nonconsensus['is_nonconsensus'] = is_nonconsensus
    
    # Set nonconsensus phenotype from specified column
    if nonconsensus_col in df_nonconsensus.columns:
        df_nonconsensus['nonconsensus_phenotype'] = df_nonconsensus[nonconsensus_col].where(
            is_nonconsensus, other=None
        )
    else:
        print(f"  WARNING: Non-consensus phenotype column '{nonconsensus_col}' not found")
        df_nonconsensus['nonconsensus_phenotype'] = None
    
    nonconsensus_rate = (df_nonconsensus['is_nonconsensus'].sum() / len(df_nonconsensus)) * 100
    print(f"  Found {df_nonconsensus['is_nonconsensus'].sum()} non-consensus cells ({nonconsensus_rate:.1f}%)")
    
    return df_nonconsensus


def calculate_consensus_performance(df_consensus):
    """Calculate performance metrics for consensus predictions."""
    print("Calculating consensus performance metrics...")

    consensus_df = df_consensus[
        df_consensus['is_consensus'] &
        df_consensus['ground_truth'].notna() &
        df_consensus['consensus_phenotype'].notna()
    ].copy()

    if len(consensus_df) == 0:
        print("  No valid consensus cells found for performance calculation!")
        return None

    y_true = consensus_df['ground_truth']
    y_pred = consensus_df['consensus_phenotype']

    labels = sorted(list(set(y_true) | set(y_pred)))
    precision_per, recall_per, f1_per, support_per = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=0
    )
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    results = {
        'method': 'Consensus',
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_weighted': precision_w,
        'recall_weighted': recall_w,
        'f1_weighted': f1_w,
        'cohen_kappa': cohen_kappa_score(y_true, y_pred),
        'matthews_corr': matthews_corrcoef(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels),
        'labels': labels,
        'per_class_precision': precision_per,
        'per_class_recall': recall_per,
        'per_class_f1': f1_per,
        'per_class_support': support_per,
        'n_cells': len(consensus_df),
        'coverage': (len(consensus_df) / len(df_consensus)) * 100
    }

    print(f"  Consensus accuracy: {results['accuracy']:.3f}")
    print(f"  Consensus F1-score: {results['f1_weighted']:.3f}")
    print(f"  Cohen's Kappa: {results['cohen_kappa']:.3f}")
    
    return results


def calculate_nonconsensus_performance(df_nonconsensus):
    """Calculate performance metrics for non-consensus predictions."""
    print("Calculating non-consensus performance metrics...")

    nonconsensus_df = df_nonconsensus[
        df_nonconsensus['is_nonconsensus'] &
        df_nonconsensus['ground_truth'].notna() &
        df_nonconsensus['nonconsensus_phenotype'].notna()
    ].copy()

    if len(nonconsensus_df) == 0:
        print("  No valid non-consensus cells found for performance calculation!")
        return None

    y_true = nonconsensus_df['ground_truth']
    y_pred = nonconsensus_df['nonconsensus_phenotype']

    labels = sorted(list(set(y_true) | set(y_pred)))
    precision_per, recall_per, f1_per, support_per = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=0
    )
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    results = {
        'method': 'Non-Consensus',
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_weighted': precision_w,
        'recall_weighted': recall_w,
        'f1_weighted': f1_w,
        'cohen_kappa': cohen_kappa_score(y_true, y_pred),
        'matthews_corr': matthews_corrcoef(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels),
        'labels': labels,
        'per_class_precision': precision_per,
        'per_class_recall': recall_per,
        'per_class_f1': f1_per,
        'per_class_support': support_per,
        'n_cells': len(nonconsensus_df),
        'coverage': (len(nonconsensus_df) / len(df_nonconsensus)) * 100
    }

    print(f"  Non-consensus accuracy: {results['accuracy']:.3f}")
    print(f"  Non-consensus F1-score: {results['f1_weighted']:.3f}")
    print(f"  Cohen's Kappa: {results['cohen_kappa']:.3f}")
    
    return results


# =============================================================================
# VISUALIZATIONS
# =============================================================================
def create_comprehensive_performance_comparison(all_results):
    print("Creating comprehensive performance comparison...")
    if not all_results:
        print("  No results to visualize!")
        return
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(22, 14))
    fig.suptitle('Comprehensive Phenotyping Validation: Individual Methods + Consensus',
                 fontsize=16, fontweight='bold')
    
    methods = [r['method'] for r in all_results]
    accuracies = [r['accuracy'] for r in all_results]
    
    colors = ['red' if m == 'Consensus' else 'skyblue' for m in methods]
    bars = ax1.bar(methods, accuracies, color=colors, alpha=0.8, edgecolor='black')
    ax1.set_title('Overall Accuracy Comparison', fontweight='bold')
    ax1.set_ylabel('Accuracy', fontweight='bold')
    ax1.set_ylim(0, 1)
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
    
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Combined Multiple Metrics + Cohen's Kappa in one figure
    metrics = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'cohen_kappa']
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', "Cohen's Kappa"]
    x = np.arange(len(methods))
    width = 0.18
    
    bars_list = []
    for i, (metric, name) in enumerate(zip(metrics, metric_names)):
        values = [r[metric] for r in all_results]
        bars = ax2.bar(x + i*width - 2*width, values, width, label=name, alpha=0.85)
        bars_list.append(bars)
    
    ax2.set_title('Multiple Metrics Comparison (Including Cohen\'s Kappa)', fontweight='bold')
    ax2.set_ylabel('Score', fontweight='bold')
    ax2.set_xlabel('Methods', fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(methods, rotation=45, ha='right')
    ax2.set_ylim(0, 1.05)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')
    # Move legend outside the plot area to avoid overlapping
    ax2.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=True, fontsize=10, edgecolor='black')
    
    # Number of Cells Analyzed
    n_cells = [r['n_cells'] for r in all_results]
    colors = ['red' if m == 'Consensus' else 'orange' for m in methods]
    bars = ax3.bar(methods, n_cells, color=colors, alpha=0.8, edgecolor='black')
    ax3.set_title('Number of Cells Analyzed', fontweight='bold')
    ax3.set_ylabel('Number of Cells', fontweight='bold')
    ax3.set_xlabel('Methods', fontweight='bold')
    plt.setp(ax3.get_xticklabels(), rotation=45, ha='right')
    ax3.grid(axis='y', alpha=0.3, linestyle='--')
    
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 100,
                 f'{int(height):,}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Hide the fourth subplot
    ax4.axis('off')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(OUTPUT_DIR, 'comprehensive_performance_comparison.png'),
                dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    plt.close()


def create_spider_web_performance_comparison(all_results):
    """
    Creates a publication-quality radar/spider chart for multi-dimensional 
    performance comparison with an improved layout and color scheme.
    """
    print("Creating publication-quality spider web performance comparison...")
    if not all_results:
        print("   No results to visualize!")
        return
    
    # === 1. Data & Metric Preparation ===
    metrics = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'cohen_kappa']
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', "Cohen's Kappa"]
    methods = [r['method'] for r in all_results]
    
    # === 2. Define a professional color palette ===
    colors = [
        '#1f77b4',  # Muted Blue
        '#ff7f0e',  # Safety Orange
        '#2ca02c',  # Cooked Asparagus Green
        '#d62728',  # Brick Red
        '#9467bd',  # Muted Purple
        '#8c564b',  # Chestnut Brown
    ]
    
    # === 3. Set up the plot aesthetics ===
    num_vars = len(metric_labels)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    
    # === 4. Plotting each method's performance ===
    for idx, result in enumerate(all_results):
        values = [result.get(metric, 0) for metric in metrics]
        values[metrics.index('cohen_kappa')] = (values[metrics.index('cohen_kappa')] + 1) / 2
        values += values[:1]
        
        color = colors[idx % len(colors)]
        
        ax.plot(angles, values, 'o-', color=color, linewidth=2.5, markersize=7, label=result['method'])
        ax.fill(angles, values, color=color, alpha=0.2)

    # === 5. Polishing the plot for publication ===
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], color="grey", size=10)
    
    ax.grid(color='grey', linestyle='--', linewidth=0.5)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_labels, size=12)

    # FINAL ADJUSTMENT: Increase padding from 15 to 25
    ax.tick_params(axis='x', pad=25) 

    for label, angle in zip(ax.get_xticklabels(), angles):
        if angle in (0, np.pi):
            label.set_horizontalalignment('center')
        elif 0 < angle < np.pi:
            label.set_horizontalalignment('left')
        else:
            label.set_horizontalalignment('right')
    
    ax.set_rlabel_position(180 / num_vars)

    # === 6. Title, Legend, and Footnote ===
    plt.suptitle('Multi-Dimensional Performance Comparison', fontsize=18, fontweight='bold', y=1.02)
    
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False, fontsize=11)

    fig.text(0.99, 0.01, "Note: Cohen's Kappa is scaled from [-1,1] to [0,1] for visualization.",
             ha='right', va='bottom', fontsize=9, color='gray', style='italic')

    # === 7. Save the Figure ===
    plt.tight_layout(pad=2)
    plt.savefig(
        os.path.join(OUTPUT_DIR, 'spider_web_performance_comparison_revised.png'),
        dpi=FIGURE_DPI,
        bbox_inches='tight'
    )
    plt.show()
    plt.close()


def create_comprehensive_confusion_matrices(all_results):
    print("Creating comprehensive confusion matrices...")
    if not all_results:
        return
    
    n_methods = len(all_results)
    n_cols = 2
    n_rows = (n_methods + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 8 * n_rows), squeeze=False)
    axes = axes.flatten()

    fig.suptitle('Confusion Matrices: All Methods + Consensus',
                 fontsize=16, fontweight='bold')
    
    for i, result in enumerate(all_results):
        ax = axes[i]
        cm = result['confusion_matrix']
        labels = result['labels']
        
        im = ax.imshow(cm, cmap='Blues')
        
        for i_cm in range(len(labels)):
            for j_cm in range(len(labels)):
                ax.text(j_cm, i_cm, cm[i_cm, j_cm],
                        ha="center", va="center", color="black", fontweight='bold')
        
        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right')
        ax.set_yticklabels(labels)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        
        title_color = 'red' if result['method'] == 'Consensus' else 'black'
        ax.set_title(f'{result["method"]}\n(Acc: {result["accuracy"]:.3f})',
                     color=title_color, fontweight='bold')
        
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    for i in range(n_methods, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(OUTPUT_DIR, 'comprehensive_confusion_matrices.png'),
                dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    plt.close()


def create_comprehensive_per_class_analysis(all_results):
    print("Creating comprehensive per-class analysis...")
    if not all_results:
        print("  No results to visualize!")
        return
    
    # Get all cell types and filter out specified phenotypes
    all_cell_types = sorted(list(set(l for r in all_results for l in r['labels'])))
    filtered_cell_types = [ct for ct in all_cell_types if ct not in FILTER_OUT_PHENOTYPES]
    
    # Create matrices using filtered cell types only
    methods = [r['method'] for r in all_results]
    f1_matrix = pd.DataFrame(index=filtered_cell_types, columns=methods, dtype=float)
    precision_matrix = pd.DataFrame(index=filtered_cell_types, columns=methods, dtype=float)
    recall_matrix = pd.DataFrame(index=filtered_cell_types, columns=methods, dtype=float)
    
    # Populate matrices only for filtered cell types
    for result in all_results:
        method = result['method']
        for i, label in enumerate(result['labels']):
            if label in filtered_cell_types:
                f1_matrix.loc[label, method] = result['per_class_f1'][i]
                precision_matrix.loc[label, method] = result['per_class_precision'][i]
                recall_matrix.loc[label, method] = result['per_class_recall'][i]
    
    # Adjust figure size based on filtered cell types
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, max(8, len(filtered_cell_types) * 0.5)))
    fig.suptitle(f'Per-Class Performance: All Methods + Consensus (Filtered: {len(filtered_cell_types)} cell types)',
                 fontsize=16, fontweight='bold')
    
    sns.heatmap(precision_matrix, annot=True, fmt='.2f', cmap='Reds', ax=ax1,
                cbar_kws={'label': 'Precision'}, vmin=0, vmax=1, 
                annot_kws={'size': 14})
    ax1.set_title('Precision by Cell Type')
    
    sns.heatmap(recall_matrix, annot=True, fmt='.2f', cmap='Greens', ax=ax2,
                cbar_kws={'label': 'Recall'}, vmin=0, vmax=1,
                annot_kws={'size': 14})
    ax2.set_title('Recall by Cell Type')
    
    sns.heatmap(f1_matrix, annot=True, fmt='.2f', cmap='Blues', ax=ax3,
                cbar_kws={'label': 'F1-Score'}, vmin=0, vmax=1,
                annot_kws={'size': 14})
    ax3.set_title('F1-Score by Cell Type')
    
    for ax in [ax1, ax2, ax3]:
        ax.set_xlabel('Method')
        ax.set_ylabel('Cell Type' if ax == ax1 else '')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(OUTPUT_DIR, 'comprehensive_per_class_analysis.png'),
                dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    plt.close()


def create_average_per_class_metrics_analysis(all_results):
    """
    Creates a figure showing average Precision, Recall, and F1-Score across all cell types
    for each phenotyping method, with error bars representing standard deviation.
    """
    print("Creating average per-class metrics analysis with error bars...")
    if not all_results:
        print("  No results to visualize!")
        return
    
    # Get all cell types and filter out specified phenotypes
    all_cell_types = sorted(list(set(l for r in all_results for l in r['labels'])))
    filtered_cell_types = [ct for ct in all_cell_types if ct not in FILTER_OUT_PHENOTYPES]
    
    # Collect methods and initialize storage
    methods = [r['method'] for r in all_results]
    
    # Store metrics for each method
    method_metrics = {method: {'precision': [], 'recall': [], 'f1': []} for method in methods}
    
    # Populate metrics by extracting per-class values for filtered cell types
    for result in all_results:
        method = result['method']
        for i, label in enumerate(result['labels']):
            if label in filtered_cell_types:
                method_metrics[method]['precision'].append(result['per_class_precision'][i])
                method_metrics[method]['recall'].append(result['per_class_recall'][i])
                method_metrics[method]['f1'].append(result['per_class_f1'][i])
    
    # Calculate mean and std for each method
    metric_stats = {}
    for method in methods:
        metric_stats[method] = {
            'precision_mean': np.mean(method_metrics[method]['precision']) if method_metrics[method]['precision'] else 0,
            'precision_std': np.std(method_metrics[method]['precision']) if method_metrics[method]['precision'] else 0,
            'recall_mean': np.mean(method_metrics[method]['recall']) if method_metrics[method]['recall'] else 0,
            'recall_std': np.std(method_metrics[method]['recall']) if method_metrics[method]['recall'] else 0,
            'f1_mean': np.mean(method_metrics[method]['f1']) if method_metrics[method]['f1'] else 0,
            'f1_std': np.std(method_metrics[method]['f1']) if method_metrics[method]['f1'] else 0,
        }
    
    # Create figure with three subplots
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f'Average Per-Class Metrics Across Methods (Filtered: {len(filtered_cell_types)} cell types)',
                 fontsize=16, fontweight='bold')
    
    x = np.arange(len(methods))
    width = 0.6
    
    # Plot 1: Average Precision with error bars
    precision_means = [metric_stats[m]['precision_mean'] for m in methods]
    precision_stds = [metric_stats[m]['precision_std'] for m in methods]
    bars1 = ax1.bar(x, precision_means, width, yerr=precision_stds, capsize=8, 
                    alpha=0.8, edgecolor='black', linewidth=1.5, color='coral', error_kw={'linewidth': 2})
    ax1.set_ylabel('Average Precision', fontweight='bold', fontsize=12)
    ax1.set_xlabel('Methods', fontweight='bold', fontsize=12)
    ax1.set_title('Average Precision by Cell Type', fontweight='bold', fontsize=13)
    ax1.set_xticks(x)
    ax1.set_xticklabels(methods, rotation=45, ha='right')
    ax1.set_ylim(0, 1.1)
    ax1.grid(axis='y', alpha=0.3, linestyle='--')
    
    for i, (mean, std) in enumerate(zip(precision_means, precision_stds)):
        ax1.text(i, mean + std + 0.02, f'{mean:.3f}', ha='center', va='bottom', 
                fontweight='bold', fontsize=10)
    
    # Plot 2: Average Recall with error bars
    recall_means = [metric_stats[m]['recall_mean'] for m in methods]
    recall_stds = [metric_stats[m]['recall_std'] for m in methods]
    bars2 = ax2.bar(x, recall_means, width, yerr=recall_stds, capsize=8, 
                    alpha=0.8, edgecolor='black', linewidth=1.5, color='lightgreen', error_kw={'linewidth': 2})
    ax2.set_ylabel('Average Recall', fontweight='bold', fontsize=12)
    ax2.set_xlabel('Methods', fontweight='bold', fontsize=12)
    ax2.set_title('Average Recall by Cell Type', fontweight='bold', fontsize=13)
    ax2.set_xticks(x)
    ax2.set_xticklabels(methods, rotation=45, ha='right')
    ax2.set_ylim(0, 1.1)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')
    
    for i, (mean, std) in enumerate(zip(recall_means, recall_stds)):
        ax2.text(i, mean + std + 0.02, f'{mean:.3f}', ha='center', va='bottom', 
                fontweight='bold', fontsize=10)
    
    # Plot 3: Average F1-Score with error bars
    f1_means = [metric_stats[m]['f1_mean'] for m in methods]
    f1_stds = [metric_stats[m]['f1_std'] for m in methods]
    bars3 = ax3.bar(x, f1_means, width, yerr=f1_stds, capsize=8, 
                    alpha=0.8, edgecolor='black', linewidth=1.5, color='skyblue', error_kw={'linewidth': 2})
    ax3.set_ylabel('Average F1-Score', fontweight='bold', fontsize=12)
    ax3.set_xlabel('Methods', fontweight='bold', fontsize=12)
    ax3.set_title('Average F1-Score by Cell Type', fontweight='bold', fontsize=13)
    ax3.set_xticks(x)
    ax3.set_xticklabels(methods, rotation=45, ha='right')
    ax3.set_ylim(0, 1.1)
    ax3.grid(axis='y', alpha=0.3, linestyle='--')
    
    for i, (mean, std) in enumerate(zip(f1_means, f1_stds)):
        ax3.text(i, mean + std + 0.02, f'{mean:.3f}', ha='center', va='bottom', 
                fontweight='bold', fontsize=10)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(OUTPUT_DIR, 'average_per_class_metrics_with_errorbars.png'),
                dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    plt.close()


def create_method_agreement_analysis(df_consensus):
    print("Creating method agreement analysis...")
    if 'is_consensus' not in df_consensus.columns:
        return
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Method Agreement Analysis', fontsize=16, fontweight='bold')
    
    # Plot 1: Consensus/Non-consensus Rate by Dataset
    if 'dataset_name' in df_consensus.columns:
        consensus_by_dataset = df_consensus.groupby('dataset_name')['is_consensus'].value_counts(normalize=True).unstack().fillna(0)
        consensus_rate = consensus_by_dataset.get(True, 0) * 100
        
        bars = ax1.bar(consensus_rate.index, consensus_rate.values,
                       color=sns.color_palette("Set2", len(consensus_rate)))
        ax1.set_title('Consensus Rate by Dataset')
        ax1.set_ylabel('Consensus Rate (%)')
        ax1.set_ylim(0, 100)
        plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 1,
                     f'{height:.1f}%', ha='center', va='bottom')
    
    # Plot 2: Distribution of Consensus Strength
    consensus_cells_df = df_consensus[df_consensus['is_consensus']]
    if not consensus_cells_df.empty:
        consensus_counts = consensus_cells_df['consensus_count'].value_counts().sort_index()
        ax2.bar(consensus_counts.index, consensus_counts.values,
                color=sns.color_palette("viridis", len(consensus_counts)))
        ax2.set_title('Distribution of Consensus Strength')
        ax2.set_xlabel('Number of Methods Agreeing')
        ax2.set_ylabel('Number of Cells')
        for count, cells in consensus_counts.items():
            ax2.text(count, cells, str(cells), ha='center', va='bottom')
    
    # Plot 3: Cell Type Distribution (Consensus vs Non-consensus)
    consensus_gt_counts = consensus_cells_df['ground_truth'].value_counts() if not consensus_cells_df.empty else pd.Series()
    
    # Add non-consensus distribution if available
    if 'is_nonconsensus' in df_consensus.columns:
        nonconsensus_cells_df = df_consensus[df_consensus['is_nonconsensus']]
        nonconsensus_gt_counts = nonconsensus_cells_df['ground_truth'].value_counts() if not nonconsensus_cells_df.empty else pd.Series()
        
        dist_df = pd.DataFrame({
            'Consensus': consensus_gt_counts,
            'Non-consensus': nonconsensus_gt_counts
        }).fillna(0).astype(int)
        
        ax3.set_title('Cell Type Distribution: Consensus vs Non-consensus')
    else:
        consensus_pred_counts = consensus_cells_df['consensus_phenotype'].value_counts() if not consensus_cells_df.empty else pd.Series()
        
        dist_df = pd.DataFrame({
            'Ground Truth': consensus_gt_counts,
            'Consensus': consensus_pred_counts
        }).fillna(0).astype(int)
        
        ax3.set_title('Cell Type Distribution: GT vs Consensus')
    
    if not dist_df.empty:
        dist_df.plot(kind='bar', ax=ax3, width=0.8)
        ax3.set_ylabel('Number of Cells')
        ax3.tick_params(axis='x', rotation=45)
        ax3.legend()

    # Plot 4: Summary Text
    total_cells = len(df_consensus)
    num_consensus_cells = df_consensus['is_consensus'].sum()
    consensus_rate = (num_consensus_cells / total_cells) * 100 if total_cells > 0 else 0
    
    # Add non-consensus info if available
    num_nonconsensus_cells = 0
    nonconsensus_rate = 0
    if 'is_nonconsensus' in df_consensus.columns:
        num_nonconsensus_cells = df_consensus['is_nonconsensus'].sum()
        nonconsensus_rate = (num_nonconsensus_cells / total_cells) * 100 if total_cells > 0 else 0
    
    summary_text = (
        f"VALIDATION SUMMARY\n\n"
        f"Total Cells: {total_cells:,}\n"
        f"Consensus Cells: {num_consensus_cells:,} ({consensus_rate:.1f}%)\n"
    )
    
    if num_nonconsensus_cells > 0:
        summary_text += f"Non-consensus Cells: {num_nonconsensus_cells:,} ({nonconsensus_rate:.1f}%)\n"
        if NONCONSENSUS_SCORE_RANGE:
            summary_text += f"Score Range: {NONCONSENSUS_SCORE_RANGE}\n"
    
    summary_text += f"\nMethods Analyzed: {len(PHENOTYPING_METHODS)}\n"
    summary_text += f"Min Methods for Consensus: {MIN_CONSENSUS_METHODS}"
    
    ax4.text(0.05, 0.5, summary_text, fontsize=10, va='center',
             bbox=dict(boxstyle="round,pad=0.5", fc="lightblue", alpha=0.7))
    ax4.axis('off')
    ax4.set_title('Analysis Summary')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(OUTPUT_DIR, 'method_agreement_analysis.png'),
                dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    plt.close()


# =============================================================================
# CSV AND REPORT GENERATION
# =============================================================================
def generate_csv_outputs(df_consensus, all_results):
    print("Generating CSV outputs...")
    
    # Overall performance comparison
    comparison_df = pd.DataFrame(all_results).drop(columns=[
        'confusion_matrix', 'labels', 'per_class_precision', 'per_class_recall',
        'per_class_f1', 'per_class_support'
    ])
    comparison_df.sort_values('accuracy', ascending=False, inplace=True)
    comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'performance_comparison.csv'), index=False)
    print("  Performance comparison saved.")
    
    # Per-class performance
    per_class_data = []
    for result in all_results:
        for i, cell_type in enumerate(result['labels']):
            per_class_data.append({
                'Method': result['method'],
                'Cell_Type': cell_type,
                'Precision': result['per_class_precision'][i],
                'Recall': result['per_class_recall'][i],
                'F1_Score': result['per_class_f1'][i],
                'Support': result['per_class_support'][i]
            })
    per_class_df = pd.DataFrame(per_class_data)
    per_class_df.to_csv(os.path.join(OUTPUT_DIR, 'per_class_performance.csv'), index=False)
    print("  Per-class performance saved.")

    # Detailed consensus cell data
    if 'is_consensus' in df_consensus.columns:
        consensus_cells = df_consensus[df_consensus['is_consensus']].copy()
        if not consensus_cells.empty:
            output_cols = ['dataset_name', 'cell_id', 'x_centroid', 'y_centroid',
                           'ground_truth', 'consensus_phenotype', 'consensus_count',
                           'consensus_methods'] + PHENOTYPING_METHODS
            available_cols = [col for col in output_cols if col in consensus_cells.columns]
            
            consensus_detailed = consensus_cells[available_cols].copy()
            consensus_detailed['consensus_correct'] = (
                consensus_detailed['ground_truth'] == consensus_detailed['consensus_phenotype']
            )
            consensus_detailed.to_csv(os.path.join(OUTPUT_DIR, 'detailed_consensus_cells.csv'), index=False)
            print(f"  Detailed consensus cells saved: {len(consensus_detailed)} cells")

    # Detailed non-consensus cell data
    if 'is_nonconsensus' in df_consensus.columns:
        nonconsensus_cells = df_consensus[df_consensus['is_nonconsensus']].copy()
        if not nonconsensus_cells.empty:
            output_cols = ['dataset_name', 'cell_id', 'x_centroid', 'y_centroid',
                           'ground_truth', 'nonconsensus_phenotype'] + PHENOTYPING_METHODS
            
            # Add ML consensus score if available
            if 'ml_consensus_score' in nonconsensus_cells.columns:
                output_cols.append('ml_consensus_score')
            
            available_cols = [col for col in output_cols if col in nonconsensus_cells.columns]
            
            nonconsensus_detailed = nonconsensus_cells[available_cols].copy()
            nonconsensus_detailed['nonconsensus_correct'] = (
                nonconsensus_detailed['ground_truth'] == nonconsensus_detailed['nonconsensus_phenotype']
            )
            nonconsensus_detailed.to_csv(os.path.join(OUTPUT_DIR, 'detailed_nonconsensus_cells.csv'), index=False)
            print(f"  Detailed non-consensus cells saved: {len(nonconsensus_detailed)} cells")


def generate_report(df_consensus, all_results):
    print("Generating analysis report...")
    report_path = os.path.join(OUTPUT_DIR, 'analysis_report.txt')
    
    with open(report_path, 'w') as f:
        f.write("PHENOTYPING VALIDATION REPORT\n")
        f.write("="*80 + "\n\n")
        
        f.write("ANALYSIS SUMMARY\n")
        f.write("-"*40 + "\n")
        f.write(f"Total cells analyzed: {len(df_consensus)}\n")
        f.write(f"Methods: {', '.join(PHENOTYPING_METHODS)}\n")

        if 'is_consensus' in df_consensus.columns:
            consensus_cells = df_consensus['is_consensus'].sum()
            consensus_rate = (consensus_cells / len(df_consensus)) * 100 if len(df_consensus) > 0 else 0
            f.write(f"Consensus cells found: {consensus_cells} ({consensus_rate:.1f}%)\n")
            f.write(f"Minimum methods for consensus: {MIN_CONSENSUS_METHODS}\n\n")

        if not all_results:
            f.write("No results to report.\n")
            return
            
        sorted_results = sorted(all_results, key=lambda x: x['accuracy'], reverse=True)
        
        f.write("PERFORMANCE RANKING (BY ACCURACY)\n")
        f.write("-"*40 + "\n")
        for i, result in enumerate(sorted_results, 1):
            f.write(f"{i}. {result['method']:<20} Accuracy: {result['accuracy']:.3f}\n")
        f.write("\n")
        
        f.write("DETAILED PERFORMANCE METRICS\n")
        f.write("-"*50 + "\n")
        for result in sorted_results:
            f.write(f"\n--- {result['method'].upper()} ---\n")
            f.write(f"  Accuracy:             {result['accuracy']:.3f}\n")
            f.write(f"  F1-Score (weighted):  {result['f1_weighted']:.3f}\n")
            f.write(f"  Precision (weighted): {result['precision_weighted']:.3f}\n")
            f.write(f"  Recall (weighted):    {result['recall_weighted']:.3f}\n")
            f.write(f"  Cohen's Kappa:        {result['cohen_kappa']:.3f}\n")
            f.write(f"  Matthews Corr Coef:   {result['matthews_corr']:.3f}\n")
            f.write(f"  Cells analyzed:       {result['n_cells']}\n")
            f.write(f"  Coverage:             {result['coverage']:.1f}%\n")
    
    print(f"  Analysis report saved: {report_path}")
    

# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    print("="*80)
    print("STARTING COMPREHENSIVE PHENOTYPING VALIDATION ANALYSIS")
    print("="*80)
    
    print("\nStep 1: Loading and preparing data...")
    df = prepare_data_for_analysis(INPUT_FILE)
    if df is None:
        print("Analysis halted: Could not load data.")
        return
    
    print("\nStep 2: Validating individual phenotyping methods...")
    individual_results = validate_individual_methods(df, PHENOTYPING_METHODS)
    if not individual_results:
        print("Analysis halted: No individual method results were generated.")
        return
        
    # Consensus and Non-consensus analysis based on user settings
    all_results = individual_results.copy()
    df_consensus = df  # Default to original dataframe
    df_nonconsensus = df  # Default to original dataframe
    consensus_results = None
    nonconsensus_results = None
    
    # Step 3: Consensus Analysis
    if INCLUDE_CONSENSUS_ANALYSIS and len(PHENOTYPING_METHODS) >= MIN_CONSENSUS_METHODS:
        print("\nStep 3: Identifying consensus cells...")
        df_consensus = identify_consensus_cells(df, PHENOTYPING_METHODS, MIN_CONSENSUS_METHODS)
        
        print("\nStep 4: Calculating consensus performance...")
        consensus_results = calculate_consensus_performance(df_consensus)
        
        if consensus_results and not CONSENSUS_SEPARATE_ANALYSIS:
            all_results.append(consensus_results)
            print(f"  Consensus results added to main analysis")
        elif consensus_results and CONSENSUS_SEPARATE_ANALYSIS:
            print(f"  Consensus results will be analyzed separately")
        
    elif INCLUDE_CONSENSUS_ANALYSIS and len(PHENOTYPING_METHODS) < MIN_CONSENSUS_METHODS:
        print(f"\nSkipping consensus analysis: Only {len(PHENOTYPING_METHODS)} methods provided, need at least {MIN_CONSENSUS_METHODS}")
    else:
        print(f"\nSkipping consensus analysis (INCLUDE_CONSENSUS_ANALYSIS = {INCLUDE_CONSENSUS_ANALYSIS})")
    
    # Step 3b: Non-consensus Analysis
    if INCLUDE_NONCONSENSUS_ANALYSIS and 'is_consensus' in df_consensus.columns:
        print("\nStep 3b: Identifying non-consensus cells...")
        df_nonconsensus = identify_nonconsensus_cells(
            df_consensus, 
            NONCONSENSUS_PHENOTYPE_COLUMN, 
            NONCONSENSUS_SCORE_RANGE
        )
        
        print("\nStep 4b: Calculating non-consensus performance...")
        nonconsensus_results = calculate_nonconsensus_performance(df_nonconsensus)
        
        if nonconsensus_results and not NONCONSENSUS_SEPARATE_ANALYSIS:
            all_results.append(nonconsensus_results)
            print(f"  Non-consensus results added to main analysis")
        elif nonconsensus_results and NONCONSENSUS_SEPARATE_ANALYSIS:
            print(f"  Non-consensus results will be analyzed separately")
            
    elif INCLUDE_NONCONSENSUS_ANALYSIS and 'is_consensus' not in df_consensus.columns:
        print(f"\nSkipping non-consensus analysis: Consensus analysis must be enabled first")
    else:
        print(f"\nSkipping non-consensus analysis (INCLUDE_NONCONSENSUS_ANALYSIS = {INCLUDE_NONCONSENSUS_ANALYSIS})")
    
    print(f"  Total result sets to process: {len(all_results)}")
    
    print("\nStep 5: Generating visualizations...")
    create_comprehensive_performance_comparison(all_results)
    create_spider_web_performance_comparison(all_results)
    create_comprehensive_confusion_matrices(all_results)
    create_comprehensive_per_class_analysis(all_results)
    create_average_per_class_metrics_analysis(all_results)
    create_method_agreement_analysis(df_consensus)
    
    print("\nStep 6: Generating CSV outputs...")
    generate_csv_outputs(df_consensus, all_results)
    
    print("\nStep 7: Generating final report...")
    generate_report(df_consensus, all_results)
    
    # If consensus analysis was run separately, generate separate consensus report
    if INCLUDE_CONSENSUS_ANALYSIS and CONSENSUS_SEPARATE_ANALYSIS and consensus_results:
        print("\nStep 8: Generating separate consensus analysis...")
        consensus_only_results = [consensus_results]
        
        # Create separate consensus visualizations
        create_comprehensive_performance_comparison(consensus_only_results)
        create_spider_web_performance_comparison(consensus_only_results)
        create_comprehensive_confusion_matrices(consensus_only_results)
        create_comprehensive_per_class_analysis(consensus_only_results)
        
        # Save consensus-only results
        consensus_comparison_df = pd.DataFrame(consensus_only_results).drop(columns=[
            'confusion_matrix', 'labels', 'per_class_precision', 'per_class_recall',
            'per_class_f1', 'per_class_support'
        ])
        consensus_comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'consensus_only_performance.csv'), index=False)
        
        print(f"  Separate consensus analysis complete")
    
    # If non-consensus analysis was run separately, generate separate non-consensus report
    if INCLUDE_NONCONSENSUS_ANALYSIS and NONCONSENSUS_SEPARATE_ANALYSIS and nonconsensus_results:
        print("\nStep 8b: Generating separate non-consensus analysis...")
        nonconsensus_only_results = [nonconsensus_results]
        
        # Create separate non-consensus visualizations
        create_comprehensive_performance_comparison(nonconsensus_only_results)
        create_spider_web_performance_comparison(nonconsensus_only_results)
        create_comprehensive_confusion_matrices(nonconsensus_only_results)
        create_comprehensive_per_class_analysis(nonconsensus_only_results)
        
        # Save non-consensus-only results
        nonconsensus_comparison_df = pd.DataFrame(nonconsensus_only_results).drop(columns=[
            'confusion_matrix', 'labels', 'per_class_precision', 'per_class_recall',
            'per_class_f1', 'per_class_support'
        ])
        nonconsensus_comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'nonconsensus_only_performance.csv'), index=False)
        
        print(f"  Separate non-consensus analysis complete")
    
    print("\n" + "="*80)
    print("COMPREHENSIVE ANALYSIS COMPLETE")
    print(f"Results saved in: {OUTPUT_DIR}")
    if INCLUDE_CONSENSUS_ANALYSIS:
        if CONSENSUS_SEPARATE_ANALYSIS:
            print("Consensus analysis: Run separately")
        else:
            print("Consensus analysis: Included with individual methods")
    else:
        print("Consensus analysis: Skipped")
    
    if INCLUDE_NONCONSENSUS_ANALYSIS:
        if NONCONSENSUS_SEPARATE_ANALYSIS:
            print("Non-consensus analysis: Run separately")
        else:
            print("Non-consensus analysis: Included with individual methods")
        print(f"Non-consensus phenotype column: {NONCONSENSUS_PHENOTYPE_COLUMN}")
        print(f"ML consensus score range: {NONCONSENSUS_SCORE_RANGE}")
    else:
        print("Non-consensus analysis: Skipped")
    print("="*80)
if __name__ == "__main__":
    main()

In [ ]:
#F1 score (Non concensus analysis)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from collections import Counter
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
project_root = Path.cwd().parent

INPUT_FILE = project_root / 'outputs/phenotyping/all_samples_combined_corrected_phenotyped.csv'

# Output directory specifically for Non-Consensus Analysis
OUTPUT_DIR = project_root / 'outputs/phenotyping/non_consensus_analysis_only'
os.makedirs(OUTPUT_DIR, exist_ok=True)

GROUND_TRUTH_COLUMN = 'phenotype_ref1'
DATASET_ID_COLUMN = 'batch_id'

# Phenotyping methods to analyze
PHENOTYPING_METHODS = [
    'phenotype_leiden',
    'phenotype_som',
    'phenotype_threshold',
    'phenotype_scimap',
    'phenotype_ml_mr',
    'phenotype_ml_consensus',
]

# =============================================================================
# FILTER CONFIGURATION (USER REQUESTED)
# =============================================================================
# A cell is "Consensus" if >= MIN_CONSENSUS_METHODS agree.
# We want to keep cells that are NOT consensus (Agreement < MIN_CONSENSUS_METHODS)
MIN_CONSENSUS_METHODS = 3 

# Filter configuration for visualization
FILTER_OUT_PHENOTYPES = [
    'unclassified', 
    'unknown',      
]

# Figure settings
plt.style.use('default')
sns.set_palette("husl")
FIGURE_DPI = 300


# =============================================================================
# DATA PREPARATION
# =============================================================================
def prepare_data_for_analysis(input_file):
    """
    Loads and prepares the main CSV for validation analysis.
    """
    print(f"Preparing data from: {input_file}")
    try:
        df = pd.read_csv(input_file)

        if GROUND_TRUTH_COLUMN not in df.columns:
            print(f"ERROR: Ground truth column '{GROUND_TRUTH_COLUMN}' not found.")
            return None
        df.rename(columns={GROUND_TRUTH_COLUMN: 'ground_truth'}, inplace=True)

        if DATASET_ID_COLUMN not in df.columns:
            print(f"ERROR: Dataset ID column '{DATASET_ID_COLUMN}' not found.")
            return None
        df.rename(columns={DATASET_ID_COLUMN: 'dataset_name'}, inplace=True)

        # Filter out rows with invalid ground truth labels
        bad_labels = ['unclassified', ]
        df = df[~df['ground_truth'].isin(bad_labels)].copy()
        df.dropna(subset=['ground_truth'], inplace=True)

        print("Data preparation successful.")
        print(f"  Total cells initially: {len(df)}")
        return df

    except FileNotFoundError:
        print(f"ERROR: Input file not found: {input_file}")
        return None
    except Exception as e:
        print(f"ERROR during data preparation: {e}")
        return None

# =============================================================================
# NON-CONSENSUS FILTERING LOGIC
# =============================================================================
def filter_non_consensus_cells(df, methods, min_agreement):
    """
    Filters the dataframe to keep ONLY cells where fewer than 
    min_agreement methods agreed on the phenotype.
    """
    print(f"\n--- FILTERING FOR NON-CONSENSUS CELLS ---")
    print(f"Criteria: Agreement count < {min_agreement} methods")
    
    df_calc = df.copy()
    available_methods = [m for m in methods if m in df.columns]
    
    # 1. Calculate the mode (most common prediction) for each cell
    preds_df = df_calc[available_methods]
    
    # We use mode to determine what the 'majority' thought, even if it wasn't enough for consensus
    # mode(axis=1) returns a dataframe, we take column 0
    majority_phenotype = preds_df.mode(axis=1)[0]
    
    # 2. Count how many methods agreed with that majority
    agreement_count = preds_df.eq(majority_phenotype, axis=0).sum(axis=1)
    
    df_calc['agreement_count'] = agreement_count
    
    # 3. Apply the Filter: 
    # Keep row if agreement_count < MIN_CONSENSUS_METHODS
    non_consensus_mask = df_calc['agreement_count'] < min_agreement
    
    df_filtered = df_calc[non_consensus_mask].copy()
    
    n_original = len(df)
    n_filtered = len(df_filtered)
    pct = (n_filtered / n_original) * 100 if n_original > 0 else 0
    
    print(f"Original Count: {n_original}")
    print(f"Non-Consensus Count: {n_filtered} ({pct:.2f}%)")
    print(f"These are cells where {min_agreement-1} or fewer methods agreed.")
    
    return df_filtered

# =============================================================================
# INDIVIDUAL METHOD VALIDATION (Standard)
# =============================================================================
def validate_individual_methods(df, methods):
    """Validate each individual phenotyping method against ground truth."""
    print("Validating individual phenotyping methods on filtered data...")
    individual_results = []
    available_methods = [m for m in methods if m in df.columns]

    for method in available_methods:
        # print(f"  Analyzing {method}...")

        method_df = df[
            df[method].notna() & (df[method] != 'unknown') & df['ground_truth'].notna()
        ].copy()

        if len(method_df) == 0:
            print(f"    No valid predictions found for {method} in filtered set")
            continue

        y_true = method_df['ground_truth']
        y_pred = method_df[method]

        labels = sorted(list(set(y_true) | set(y_pred)))
        precision_per, recall_per, f1_per, support_per = precision_recall_fscore_support(
            y_true, y_pred, labels=labels, average=None, zero_division=0
        )
        precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0
        )

        results = {
            'method': method,
            'accuracy': accuracy_score(y_true, y_pred),
            'precision_weighted': precision_w,
            'recall_weighted': recall_w,
            'f1_weighted': f1_w,
            'cohen_kappa': cohen_kappa_score(y_true, y_pred),
            'matthews_corr': matthews_corrcoef(y_true, y_pred),
            'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels),
            'labels': labels,
            'per_class_precision': precision_per,
            'per_class_recall': recall_per,
            'per_class_f1': f1_per,
            'per_class_support': support_per,
            'n_cells': len(method_df),
            'coverage': (len(method_df) / len(df)) * 100
        }
        individual_results.append(results)
        print(f"    {method}: {results['accuracy']:.3f} accuracy")

    return individual_results

# =============================================================================
# VISUALIZATIONS (Unchanged Logic, applied to filtered data)
# =============================================================================
def create_comprehensive_performance_comparison(all_results):
    print("Creating comprehensive performance comparison...")
    if not all_results: return
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(22, 14))
    fig.suptitle(f'NON-CONSENSUS ANALYSIS (Agreement < {MIN_CONSENSUS_METHODS} methods)',
                 fontsize=16, fontweight='bold', color='darkred')
    
    methods = [r['method'] for r in all_results]
    accuracies = [r['accuracy'] for r in all_results]
    
    colors = ['skyblue' for m in methods]
    bars = ax1.bar(methods, accuracies, color=colors, alpha=0.8, edgecolor='black')
    ax1.set_title('Accuracy on Non-Consensus Cells', fontweight='bold')
    ax1.set_ylabel('Accuracy', fontweight='bold')
    ax1.set_ylim(0, 1)
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
    
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Combined Multiple Metrics
    metrics = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'cohen_kappa']
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', "Cohen's Kappa"]
    x = np.arange(len(methods))
    width = 0.18
    
    for i, (metric, name) in enumerate(zip(metrics, metric_names)):
        values = [r[metric] for r in all_results]
        ax2.bar(x + i*width - 2*width, values, width, label=name, alpha=0.85)
    
    ax2.set_title('Metrics Comparison (Non-Consensus)', fontweight='bold')
    ax2.set_ylabel('Score', fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(methods, rotation=45, ha='right')
    ax2.set_ylim(0, 1.05)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')
    ax2.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=True)
    
    # Number of Cells Analyzed
    n_cells = [r['n_cells'] for r in all_results]
    bars = ax3.bar(methods, n_cells, color='orange', alpha=0.8, edgecolor='black')
    ax3.set_title('Number of Non-Consensus Cells Analyzed', fontweight='bold')
    plt.setp(ax3.get_xticklabels(), rotation=45, ha='right')
    
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 5,
                 f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
    ax4.axis('off')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(OUTPUT_DIR, 'non_consensus_performance.png'), dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close()

def create_spider_web_performance_comparison(all_results):
    print("Creating spider web comparison...")
    if not all_results: return
    
    metrics = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'cohen_kappa']
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', "Cohen's Kappa"]
    
    # Define colors
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
    num_vars = len(metric_labels)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    
    for idx, result in enumerate(all_results):
        values = [result.get(metric, 0) for metric in metrics]
        # Normalize kappa for visualization if negative
        values[metrics.index('cohen_kappa')] = max(0, values[metrics.index('cohen_kappa')])
        values += values[:1]
        
        color = colors[idx % len(colors)]
        ax.plot(angles, values, 'o-', color=color, linewidth=2.5, label=result['method'])
        ax.fill(angles, values, color=color, alpha=0.1)

    ax.set_ylim(0, 1.0)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_labels, size=12)
    ax.tick_params(axis='x', pad=25) 
    
    plt.suptitle(f'Non-Consensus Performance (Agmt < {MIN_CONSENSUS_METHODS})', fontsize=18, fontweight='bold', y=1.02)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'non_consensus_spider_web.png'), dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close()

def create_comprehensive_confusion_matrices(all_results):
    print("Creating confusion matrices...")
    if not all_results: return
    
    n_methods = len(all_results)
    n_cols = 2
    n_rows = (n_methods + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 8 * n_rows), squeeze=False)
    axes = axes.flatten()

    fig.suptitle('Confusion Matrices: Non-Consensus Cells Only', fontsize=16, fontweight='bold')
    
    for i, result in enumerate(all_results):
        ax = axes[i]
        cm = result['confusion_matrix']
        labels = result['labels']
        
        im = ax.imshow(cm, cmap='Reds') # Changed to Reds to signify "Hard" cases
        
        for i_cm in range(len(labels)):
            for j_cm in range(len(labels)):
                ax.text(j_cm, i_cm, cm[i_cm, j_cm], ha="center", va="center", color="black", fontweight='bold')
        
        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right')
        ax.set_yticklabels(labels)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True (Ground Truth)')
        ax.set_title(f'{result["method"]}\n(Non-Consensus Acc: {result["accuracy"]:.3f})', fontweight='bold')
    
    for i in range(n_methods, len(axes)): axes[i].set_visible(False)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(OUTPUT_DIR, 'non_consensus_confusion_matrices.png'), dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close()

def create_comprehensive_per_class_analysis(all_results):
    print("Creating per-class analysis...")
    if not all_results: return
    
    all_cell_types = sorted(list(set(l for r in all_results for l in r['labels'])))
    filtered_cell_types = [ct for ct in all_cell_types if ct not in FILTER_OUT_PHENOTYPES]
    
    methods = [r['method'] for r in all_results]
    f1_matrix = pd.DataFrame(index=filtered_cell_types, columns=methods, dtype=float)
    
    for result in all_results:
        method = result['method']
        for i, label in enumerate(result['labels']):
            if label in filtered_cell_types:
                f1_matrix.loc[label, method] = result['per_class_f1'][i]
    
    plt.figure(figsize=(10, max(6, len(filtered_cell_types)*0.5)))
    sns.heatmap(f1_matrix, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1)
    plt.title(f'F1-Score by Cell Type (Non-Consensus Only)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'non_consensus_per_class_f1.png'), dpi=FIGURE_DPI)
    plt.close()

# =============================================================================
# CSV GENERATION
# =============================================================================
def generate_detailed_csv(df_filtered):
    print("Generating detailed CSV of non-consensus cells...")
    output_cols = ['dataset_name', 'cell_id', 'x_centroid', 'y_centroid', 'ground_truth', 'agreement_count'] + PHENOTYPING_METHODS
    available = [c for c in output_cols if c in df_filtered.columns]
    
    out_df = df_filtered[available].copy()
    out_path = os.path.join(OUTPUT_DIR, 'detailed_non_consensus_cells.csv')
    out_df.to_csv(out_path, index=False)
    print(f"Saved to: {out_path}")

# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    print("="*80)
    print(f"STARTING ANALYSIS: NON-CONSENSUS CELLS ONLY")
    print(f"Ground Truth: {GROUND_TRUTH_COLUMN}")
    print(f"Non-Consensus Criteria: Agreement < {MIN_CONSENSUS_METHODS} methods")
    print("="*80)
    
    # 1. Load Data
    df_raw = prepare_data_for_analysis(INPUT_FILE)
    if df_raw is None: return
    
    # 2. FILTER: Keep ONLY Non-Consensus Cells
    # This creates a subset dataframe where specific phenotyping is "hard" (disagreement)
    df_non_consensus = filter_non_consensus_cells(df_raw, PHENOTYPING_METHODS, MIN_CONSENSUS_METHODS)
    
    if len(df_non_consensus) == 0:
        print("Analysis Halted: No non-consensus cells found matching criteria.")
        return

    # 3. Analyze Methods on this SUBSET
    # We pass the filtered dataframe to the standard validation function.
    # The statistics generated (accuracy, F1, etc.) will now strictly reflect
    # performance on non-consensus cells.
    results = validate_individual_methods(df_non_consensus, PHENOTYPING_METHODS)
    
    if not results:
        print("No results generated.")
        return

    # 4. Generate Visualizations for this subset
    print("\nGenerating Figures for Non-Consensus Data...")
    create_comprehensive_performance_comparison(results)
    create_spider_web_performance_comparison(results)
    create_comprehensive_confusion_matrices(results)
    create_comprehensive_per_class_analysis(results)
    
    # 5. Save Data
    generate_detailed_csv(df_non_consensus)
    
    # Save simple performance summary
    res_df = pd.DataFrame(results)[['method', 'accuracy', 'f1_weighted', 'cohen_kappa', 'n_cells']]
    res_df.to_csv(os.path.join(OUTPUT_DIR, 'non_consensus_metrics_summary.csv'), index=False)

    print("\n" + "="*80)
    print("NON-CONSENSUS ANALYSIS COMPLETE")
    print(f"All outputs saved to: {OUTPUT_DIR}")
    print("="*80)

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, cohen_kappa_score
from sklearn.utils import resample
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from pathlib import Path
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for clean output

# Set random seed for reproducibility
np.random.seed(42)

# Define paths (as per user)
project_root = Path.cwd().parent
INPUT_FILE = project_root / 'outputs/phenotyping' / 'all_samples_combined_corrected_phenotyped.csv'

# Load data
df = pd.read_csv(INPUT_FILE)

# Define method columns and ground truths
methods = {
    'Leiden': 'phenotype_leiden',
    'SOM': 'phenotype_som',
    'Threshold': 'phenotype_threshold',
    'SciMap': 'phenotype_scimap',
    'ML_Consensus': 'phenotype_ml_consensus',
    'ML_MR': 'phenotype_ml_mr'
}
gt_columns = ['phenotype_ref1', 'phenotype_ref2']
batch_col = 'batch_id'

# Create subsets based on non-null ground truths
df_ref1 = df.dropna(subset=['phenotype_ref1']).copy()
df_ref2 = df.dropna(subset=['phenotype_ref2']).copy()

print(f"Ref1 subset size: {len(df_ref1)} cells")
print(f"Ref2 subset size: {len(df_ref2)} cells")

# Function to compute metrics for a given dataframe, method, and ground truth column
def compute_metrics(df, method_col, gt_col):
    y_true = df[gt_col]
    y_pred = df[method_col]
    # Drop rows where pred is NaN (if any)
    valid_idx = y_pred.notna()
    y_true = y_true[valid_idx]
    y_pred = y_pred[valid_idx]
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    return report, acc, kappa

# Function for bootstrap resampling and CI calculation
def bootstrap_metrics(df, method_col, gt_col, n_boot=1000, stratify_by=batch_col):
    macro_f1s = []
    for _ in range(n_boot):
        # Stratified bootstrap by batch if possible
        if stratify_by in df.columns:
            df_boot = df.groupby(stratify_by).apply(lambda x: resample(x, replace=True, n_samples=len(x))).reset_index(drop=True)
        else:
            df_boot = resample(df, replace=True, n_samples=len(df))
        report, _, _ = compute_metrics(df_boot, method_col, gt_col)
        macro_f1s.append(report['macro avg']['f1-score'])
    mean_f1 = np.mean(macro_f1s)
    ci_low, ci_high = np.percentile(macro_f1s, [2.5, 97.5])
    return mean_f1, ci_low, ci_high, macro_f1s

# Function for pairwise statistical comparison using Wilcoxon on bootstrap samples
def compare_methods(boot_f1_a, boot_f1_b, method_a, method_b):
    stat, p = wilcoxon(boot_f1_a, boot_f1_b)
    return {'Method A': method_a, 'Method B': method_b, 'p-value': p, 'Significant (p<0.05)': p < 0.05}

# Evaluate on primary test set (ref2, independent)
print("\nEvaluating on independent test set (ref2)...")
metrics_summary_ref2 = []
boot_results_ref2 = {}
for name, col in methods.items():
    report, acc, kappa = compute_metrics(df_ref2, col, 'phenotype_ref2')
    mean_f1, ci_low, ci_high, boot_f1s = bootstrap_metrics(df_ref2, col, 'phenotype_ref2')
    boot_results_ref2[name] = boot_f1s
    metrics_summary_ref2.append({
        'Method': name,
        'Macro F1': report['macro avg']['f1-score'],
        'Micro F1': report['weighted avg']['f1-score'],  # Weighted is micro for F1
        'Accuracy': acc,
        'Kappa': kappa,
        'Macro F1 (95% CI)': f"{mean_f1:.3f} [{ci_low:.3f}, {ci_high:.3f}]"
    })
df_metrics_ref2 = pd.DataFrame(metrics_summary_ref2)
print(df_metrics_ref2)

# Statistical comparisons on ref2
comparisons_ref2 = []
method_names = list(methods.keys())
for a, b in combinations(method_names, 2):
    comp = compare_methods(boot_results_ref2[a], boot_results_ref2[b], a, b)
    comparisons_ref2.append(comp)
df_comps_ref2 = pd.DataFrame(comparisons_ref2)
# Apply Bonferroni correction (divide alpha by number of tests)
n_tests = len(comparisons_ref2)
df_comps_ref2['Adjusted Significant'] = df_comps_ref2['p-value'] < (0.05 / n_tests)
print("\nPairwise Comparisons (Wilcoxon, Bonferroni-adjusted):")
print(df_comps_ref2)

# Per-phenotype F1 heatmap
per_class_f1 = {}
unique_phenos = sorted(df_ref2['phenotype_ref2'].unique())
for name, col in methods.items():
    report, _, _ = compute_metrics(df_ref2, col, 'phenotype_ref2')
    per_class_f1[name] = [report[pheno]['f1-score'] if pheno in report else 0 for pheno in unique_phenos]
df_heatmap = pd.DataFrame(per_class_f1, index=unique_phenos)
plt.figure(figsize=(10, 8))
sns.heatmap(df_heatmap, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Per-Phenotype F1 Scores (Ref2)')
plt.ylabel('Phenotype')
plt.savefig('per_phenotype_f1_ref2.png', dpi=300, bbox_inches='tight')
plt.close()

# Boxplot of bootstrap macro-F1
df_boot = pd.DataFrame(boot_results_ref2).melt(var_name='Method', value_name='Macro F1')
plt.figure(figsize=(12, 6))
sns.boxplot(x='Method', y='Macro F1', data=df_boot)
plt.title('Bootstrap Distributions of Macro F1 (Ref2)')
plt.savefig('bootstrap_f1_ref2.png', dpi=300, bbox_inches='tight')
plt.close()

# Confusion matrix for top methods (example: ML_Consensus and ML_MR)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
for top_method in ['ML_Consensus', 'ML_MR']:
    y_true = df_ref2['phenotype_ref2']
    y_pred = df_ref2[methods[top_method]]
    valid_idx = y_pred.notna()
    y_true = y_true[valid_idx]
    y_pred = y_pred[valid_idx]
    cm = confusion_matrix(y_true, y_pred, labels=unique_phenos)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=unique_phenos)
    disp.plot(cmap='Blues', xticks_rotation=45)
    plt.title(f'Confusion Matrix: {top_method} (Ref2)')
    plt.savefig(f'cm_{top_method}_ref2.png', dpi=300, bbox_inches='tight')
    plt.close()

# Repeat evaluation on ref1 for completeness (noting bias for ML_MR)
print("\nEvaluating on ref1 (training set for ML_MR, potentially biased)...")
metrics_summary_ref1 = []
for name, col in methods.items():
    report, acc, kappa = compute_metrics(df_ref1, col, 'phenotype_ref1')
    metrics_summary_ref1.append({
        'Method': name,
        'Macro F1': report['macro avg']['f1-score'],
        'Micro F1': report['weighted avg']['f1-score'],
        'Accuracy': acc,
        'Kappa': kappa
    })
df_metrics_ref1 = pd.DataFrame(metrics_summary_ref1)
print(df_metrics_ref1)

# Save results to files
df_metrics_ref2.to_csv('metrics_summary_ref2.csv', index=False)
df_comps_ref2.to_csv('stat_comparisons_ref2.csv', index=False)
df_metrics_ref1.to_csv('metrics_summary_ref1.csv', index=False)

print("\nAnalysis complete. Check saved figures and CSV files for publication-ready outputs.")
print("Conclusion: Compare 'Macro F1 (95% CI)' and p-values to determine if ML_Consensus/ML_MR are comparable or superior. Non-overlapping CIs or significant p-values indicate differences.")



In [ ]:
# =============================================================================
# STATISTICAL ANALYSIS OF METHOD PERFORMANCE
# =============================================================================

import scipy.stats as stats
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

print("="*80)
print("STATISTICAL ANALYSIS OF PHENOTYPING METHOD PERFORMANCE")
print("="*80)

project_root = Path.cwd().parent
OUTPUT_DIR = project_root / 'outputs/phenotyping/comparison'
# Load the per-class performance data
per_class_csv = Path(OUTPUT_DIR) / 'per_class_performance.csv'

if not per_class_csv.exists():
    print(f"ERROR: Per-class performance file not found: {per_class_csv}")
    print("Please run the main analysis first to generate the performance data.")
else:
    # Load the data
    perf_df = pd.read_csv(per_class_csv)
    print(f"Loaded performance data: {len(perf_df)} records")
    print(f"Methods found: {perf_df['Method'].unique()}")
    print(f"Cell types found: {len(perf_df['Cell_Type'].unique())}")
    
    # Filter out 'unclassified' if present
    perf_df_filtered = perf_df[perf_df['Cell_Type'] != 'unclassified'].copy()
    print(f"After filtering unclassified: {len(perf_df_filtered)} records")
    
    # Metrics to analyze
    metrics = ['Precision', 'Recall', 'F1_Score']
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    # Statistical results storage
    stat_results = {}
    
    for i, metric in enumerate(metrics):
        ax = axes[i]
        
        print(f"\n--- Statistical Analysis for {metric} ---")
        
        # Prepare data for statistical tests
        method_groups = []
        method_names = []
        
        methods = perf_df_filtered['Method'].unique()
        for method in methods:
            method_data = perf_df_filtered[perf_df_filtered['Method'] == method][metric].values
            if len(method_data) > 0:  # Only include methods with data
                method_groups.append(method_data)
                method_names.append(method)
        
        if len(method_groups) < 2:
            print(f"Not enough methods with data for {metric}")
            continue
            
        # One-way ANOVA
        f_stat, p_value = f_oneway(*method_groups)
        
        print(f"One-way ANOVA results:")
        print(f"  F-statistic: {f_stat:.4f}")
        print(f"  P-value: {p_value:.4f}")
        
        significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
        print(f"  Significance: {significance}")
        
        # Store results
        stat_results[metric] = {
            'f_stat': f_stat,
            'p_value': p_value,
            'significance': significance,
            'methods': method_names
        }
        
        # Post-hoc analysis if ANOVA is significant
        if p_value < 0.05:
            print(f"  Performing Tukey's HSD post-hoc test...")
            
            # Prepare data for Tukey test
            tukey_data = []
            tukey_groups = []
            
            for method in method_names:
                method_values = perf_df_filtered[perf_df_filtered['Method'] == method][metric].values
                tukey_data.extend(method_values)
                tukey_groups.extend([method] * len(method_values))
            
            tukey_result = pairwise_tukeyhsd(tukey_data, tukey_groups, alpha=0.05)
            print(f"  Tukey HSD results:")
            print(tukey_result)
            
            stat_results[metric]['tukey'] = tukey_result
        
        # Create bar plot with error bars
        method_means = []
        method_stds = []
        method_labels = []
        
        for method in method_names:
            method_data = perf_df_filtered[perf_df_filtered['Method'] == method][metric].values
            method_means.append(np.mean(method_data))
            method_stds.append(np.std(method_data))
            method_labels.append(method)
        
        # Create bar plot
        bars = ax.bar(range(len(method_labels)), method_means, yerr=method_stds, 
                     capsize=5, alpha=0.8, edgecolor='black', linewidth=1)
        
        # Color bars based on performance (highest gets different color)
        best_idx = np.argmax(method_means)
        colors = ['lightcoral' if i != best_idx else 'lightgreen' for i in range(len(bars))]
        for bar, color in zip(bars, colors):
            bar.set_color(color)
        
        # Add value labels on bars
        for i, (mean, std) in enumerate(zip(method_means, method_stds)):
            ax.text(i, mean + std + 0.01, f'{mean:.3f}', 
                   ha='center', va='bottom', fontweight='bold', fontsize=10)
        
        # Customize plot
        ax.set_xlabel('Phenotyping Methods')
        ax.set_ylabel(f'{metric} Score')
        ax.set_title(f'{metric} Comparison Across Methods\n(ANOVA: F={f_stat:.2f}, p={p_value:.4f} {significance})')
        ax.set_xticks(range(len(method_labels)))
        ax.set_xticklabels(method_labels, rotation=45, ha='right')
        ax.set_ylim(0, 1.1)
        ax.grid(True, alpha=0.3)
        
        # Add significance annotations if ANOVA is significant
        if p_value < 0.05 and 'tukey' in stat_results[metric]:
            y_max = max(method_means) + max(method_stds) + 0.05
            ax.text(0.02, 0.98, f'Post-hoc: Tukey HSD\nα = 0.05', 
                   transform=ax.transAxes, va='top', ha='left',
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7),
                   fontsize=8)
    
    # Add overall summary in the fourth subplot
    ax = axes[3]
    ax.axis('off')
    
    # Create summary text
    summary_text = "STATISTICAL SUMMARY\n\n"
    
    for metric in metrics:
        if metric in stat_results:
            result = stat_results[metric]
            summary_text += f"{metric}:\n"
            summary_text += f"  F({len(result['methods'])-1}) = {result['f_stat']:.3f}\n"
            summary_text += f"  p = {result['p_value']:.4f} {result['significance']}\n"
            
            if result['p_value'] < 0.05:
                summary_text += f"  → Significant differences found\n"
                summary_text += f"  → Post-hoc analysis performed\n"
            else:
                summary_text += f"  → No significant differences\n"
            summary_text += "\n"
    
    summary_text += "Legend:\n"
    summary_text += "*** p < 0.001 (highly significant)\n"
    summary_text += "**  p < 0.01  (very significant)\n"
    summary_text += "*   p < 0.05  (significant)\n"
    summary_text += "ns  p ≥ 0.05  (not significant)\n\n"
    
    summary_text += "Green bar = Best performing method\n"
    summary_text += "Red bars = Other methods\n"
    summary_text += "Error bars = ±1 Standard Deviation"
    
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=11,
           verticalalignment='top', fontfamily='monospace',
           bbox=dict(boxstyle="round,pad=0.5", facecolor="lightblue", alpha=0.8))
    
    plt.tight_layout()
    
    # Save the figure
    stats_plot_path = Path(OUTPUT_DIR) / 'method_performance_statistical_analysis.png'
    plt.savefig(stats_plot_path, dpi=300, bbox_inches='tight')
    print(f"\nStatistical analysis plot saved to: {stats_plot_path}")
    plt.show()
    
    # Generate detailed statistical report
    report_path = Path(OUTPUT_DIR) / 'statistical_analysis_report.txt'
    with open(report_path, 'w') as f:
        f.write("STATISTICAL ANALYSIS REPORT: PHENOTYPING METHOD PERFORMANCE\n")
        f.write("="*70 + "\n\n")
        
        f.write(f"Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Data Source: {per_class_csv}\n")
        f.write(f"Total Records: {len(perf_df_filtered)}\n")
        f.write(f"Methods Analyzed: {len(method_names)}\n")
        f.write(f"Cell Types Analyzed: {len(perf_df_filtered['Cell_Type'].unique())}\n\n")
        
        for metric in metrics:
            if metric in stat_results:
                result = stat_results[metric]
                f.write(f"{metric.upper()} ANALYSIS\n")
                f.write("-" * 30 + "\n")
                f.write(f"One-way ANOVA Results:\n")
                f.write(f"  F-statistic: {result['f_stat']:.6f}\n")
                f.write(f"  P-value: {result['p_value']:.6f}\n")
                f.write(f"  Significance: {result['significance']}\n\n")
                
                # Method performance ranking
                method_scores = []
                for method in result['methods']:
                    scores = perf_df_filtered[perf_df_filtered['Method'] == method][metric].values
                    method_scores.append((method, np.mean(scores), np.std(scores)))
                
                method_scores.sort(key=lambda x: x[1], reverse=True)
                
                f.write(f"Method Performance Ranking (Mean ± SD):\n")
                for rank, (method, mean, std) in enumerate(method_scores, 1):
                    f.write(f"  {rank}. {method}: {mean:.4f} ± {std:.4f}\n")
                
                if 'tukey' in result:
                    f.write(f"\nTukey HSD Post-hoc Test Results:\n")
                    f.write(str(result['tukey']))
                    f.write("\n")
                
                f.write("\n" + "="*50 + "\n\n")
        
        f.write("INTERPRETATION GUIDELINES:\n")
        f.write("-" * 25 + "\n")
        f.write("• ANOVA tests if there are significant differences between methods\n")
        f.write("• Tukey HSD identifies which specific pairs of methods differ significantly\n")
        f.write("• Higher scores indicate better performance for all metrics\n")
        f.write("• Consider both statistical significance AND practical effect sizes\n")
        f.write("• Green bars in plots indicate the best-performing method for each metric\n")
    
    print(f"Detailed statistical report saved to: {report_path}")
    print(f"\nAnalysis complete! Check the output files for detailed results.")

## 3. Phenotype × Marker Mean Z-Score Heatmap (Selected Phenotype Column)

This section builds a heatmap of mean marker Z-scores per phenotype category for a chosen phenotype column (default: `phenotype_leiden`).

Method:
1. Reload the full unified phenotyping file (to access marker columns if an earlier step overwrote `df`).
2. Select the predefined list of phenotyping markers (batch-corrected form: `<marker>_corr`).
3. Compute Z-scores per marker across all cells: (value - global_mean) / global_std.
4. Group by the chosen phenotype column and take the mean Z-score per phenotype × marker.
5. Reorder phenotypes using a biologically meaningful fixed order.
6. Render a heatmap (markers on Y, phenotypes on X) with bwr colormap centered at 0.

Notes:
- Missing markers are silently skipped (warning printed).
- If a phenotype has no cells, it is omitted.
- Adjust `PHENOTYPE_COLUMN` to visualize other phenotype methods (`phenotype_som`, `phenotype_threshold`, `phenotype_scimap`).

In [ ]:
# =============================================================================
# Phenotype × Marker Mean Z-Score Heatmap
# =============================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Configuration
PHENOTYPE_COLUMN = 'phenotype_subsets'  # Column to display on heatmap (phenotype_subsets, phenotype_som, etc.)

# Parent phenotype filtering
PHENOTYPE_FILTER_COLUMN = 'phenotype_ml_mr'  # Column to filter by (parent phenotypes) - set to None to disable filtering
PHENOTYPE_FILTER_VALUES = ['Mac', 'Granulocyte', 'DC']  # e.g., ['Helper T', 'Treg', 'Cytotoxic T']; None uses all phenotypes
# If PHENOTYPE_FILTER_VALUES is set, heatmap will show subsets grouped by parent phenotype in that order
# If None, subsets will be sorted alphabetically by their parent phenotype

# Master marker panel (all possible markers we may attempt to include)
MARKERS = ["DAPI", "CD44", "CD31", "CD107a", "E-cadherin", "CD20", "CD4", "CD45RO",
    "Pan-Cytokeratin", "CD68", "Podoplanin", "HLA-A", "CD11c", "CD8", "HLA-DR",
    "Ki67", "CD3e", "CD69", "CD45", "CD15", "gH2AX", "FoxP3", "CD141",
    "PDGFRa", "aSMA", "CD163", "FN1", "S100A9", "CD206", "CD11b", "CD38",
    "PD-L1", "CD40", "PCNA", "Vimentin"
]

# Expected phenotype order - populated with common phenotypes
PHENOTYPE_ORDER = [
]

# Paths
project_root = Path().resolve().parent
UNIFIED_CSV = project_root / 'outputs/phenotyping' / 'all_samples_combined_corrected_phenotyped.csv'
#UNIFIED_CSV = project_root / "data/manual reference/ref3/csv/P11_roi_1.ome.tiff - Merged Image_ground_truth.csv"

if not UNIFIED_CSV.exists():
    raise FileNotFoundError(f"Unified phenotyping file not found: {UNIFIED_CSV}")

udf = pd.read_csv(UNIFIED_CSV)

# Filter for corrected==1 first
if 'corrected' in udf.columns:
    n_before = len(udf)
    udf = udf[udf['corrected'] == 1].copy()
    print(f"Filtered to corrected==1: {len(udf)} of {n_before} cells")
else:
    print("Warning: 'corrected' column not found, using all cells")

if PHENOTYPE_COLUMN not in udf.columns:
    raise ValueError(f"Column '{PHENOTYPE_COLUMN}' not in unified file. Available phenotype cols: {[c for c in udf.columns if c.startswith('phenotype_')]} ")

# Apply parent phenotype filter if specified
if PHENOTYPE_FILTER_COLUMN and PHENOTYPE_FILTER_VALUES:
    if PHENOTYPE_FILTER_COLUMN not in udf.columns:
        raise ValueError(f"Filter column '{PHENOTYPE_FILTER_COLUMN}' not found in data")
    n_before_filter = len(udf)
    udf = udf[udf[PHENOTYPE_FILTER_COLUMN].isin(PHENOTYPE_FILTER_VALUES)].copy()
    print(f"Filtered by {PHENOTYPE_FILTER_COLUMN} in {PHENOTYPE_FILTER_VALUES}: {len(udf)} of {n_before_filter} cells")
else:
    print(f"No parent filtering applied - using all {len(udf)} cells")

# Collect available corrected marker columns
corr_map = {}
available_markers = []
for m in MARKERS:
    corr_col = f"{m}"
    if corr_col in udf.columns:
        corr_map[m] = corr_col
        available_markers.append(m)
    else:
        print(f"Warning: marker column missing and skipped: {corr_col}")

if not available_markers:
    raise ValueError("None of the specified marker columns were found.")

# Subset data to phenotype + marker columns
sub_cols = [corr_map[m] for m in available_markers] + [PHENOTYPE_COLUMN]
sub_df = udf[sub_cols].copy()

# Drop rows with NA phenotype (optional)
sub_df = sub_df.dropna(subset=[PHENOTYPE_COLUMN])

# Check what phenotypes are actually present in the data
actual_phenotypes = sub_df[PHENOTYPE_COLUMN].unique()
print(f"Phenotypes found in data: {sorted(actual_phenotypes)}") 

# Compute global Z-score per marker
marker_cols = [corr_map[m] for m in available_markers]
means = sub_df[marker_cols].mean(axis=0)
stds = sub_df[marker_cols].std(axis=0).replace(0, np.nan)
z_df = (sub_df[marker_cols] - means) / stds
z_df = z_df.fillna(0)

# Append phenotype column back
z_df[PHENOTYPE_COLUMN] = sub_df[PHENOTYPE_COLUMN].values

# Aggregate mean Z per phenotype
mean_z = z_df.groupby(PHENOTYPE_COLUMN)[marker_cols].mean()

# Reindex columns to marker names for display
mean_z.index.name = 'phenotype'

# =========================================================================
# PHENOTYPE ORDERING: Group by parent phenotypes when filter applied
# =========================================================================

# Keep reference to parent phenotypes for each subset
parent_map = {}
if PHENOTYPE_FILTER_COLUMN and PHENOTYPE_FILTER_COLUMN in udf.columns:
    for subset in mean_z.index:
        matches = udf[udf[PHENOTYPE_COLUMN] == subset][PHENOTYPE_FILTER_COLUMN].unique()
        if len(matches) > 0:
            parent_map[subset] = matches[0]

if PHENOTYPE_FILTER_COLUMN and PHENOTYPE_FILTER_VALUES and parent_map:
    # Group subsets by parent phenotype in the order specified by PHENOTYPE_FILTER_VALUES
    present_phenos = []
    for parent in PHENOTYPE_FILTER_VALUES:
        parent_subsets = sorted([s for s, p in parent_map.items() if p == parent])
        if parent_subsets:
            present_phenos.extend(parent_subsets)
    
    # Add any remaining phenotypes not in specified parent list
    remaining_phenos = [p for p in sorted(mean_z.index) if p not in present_phenos]
    present_phenos.extend(remaining_phenos)
    print(f"Phenotypes ordered by parent {PHENOTYPE_FILTER_COLUMN}: {present_phenos}")
elif PHENOTYPE_FILTER_COLUMN and parent_map:
    # No filter values: sort by parent phenotype alphabetically, then subset alphabetically
    parent_to_subsets = {}
    for subset, parent in parent_map.items():
        if parent not in parent_to_subsets:
            parent_to_subsets[parent] = []
        parent_to_subsets[parent].append(subset)
    
    present_phenos = []
    for parent in sorted(parent_to_subsets.keys()):
        present_phenos.extend(sorted(parent_to_subsets[parent]))
    print(f"Phenotypes grouped by parent {PHENOTYPE_FILTER_COLUMN} (alphabetically): {present_phenos}")
else:
    # No parent column available, just sort all phenotypes
    present_phenos = sorted(mean_z.index.tolist())
    print(f"Phenotypes (alphabetically sorted): {present_phenos}")

mean_z = mean_z.loc[present_phenos]

# Rename marker columns to raw marker base names
rename_map = {corr_map[m]: m for m in available_markers}
mean_z = mean_z.rename(columns=rename_map)

# Transpose so markers on Y-axis, phenotypes on X-axis
heat_data = mean_z.T

# Apply desired marker order (keep only those present after filtering & renaming)
ordered_markers = [m for m in MARKERS if m in heat_data.index]
heat_data = heat_data.loc[ordered_markers]

print(f"Heatmap data shape: {heat_data.shape}")
print(f"Phenotypes in heatmap: {list(heat_data.columns)}") 

plt.figure(figsize=(max(10, 0.7 * heat_data.shape[1]), max(8, 0.45 * heat_data.shape[0])))
sns.heatmap(
    heat_data,
    cmap='bwr',
    center=0,
    vmin=-4, vmax=4,
    linewidths=0.5,
    linecolor='lightgray',
    cbar_kws={'label': 'Mean Z-score'},
    annot=False  # Set to True if you want values displayed in cells
)
plt.title(f'Heatmap of {PHENOTYPE_COLUMN}: Mean Marker Z-Scores', fontsize=14, pad=16)
plt.xlabel('Phenotype')
plt.ylabel('Marker')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# Save figure
OUTPUT_DIR = project_root / 'outputs/phenotyping' / 'comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
filename = f"heatmap_of_{PHENOTYPE_COLUMN}.png"
out_path = OUTPUT_DIR / filename
plt.savefig(out_path, dpi=300)
print(f"Heatmap saved to: {out_path}")
print(f"Markers displayed (in order): {ordered_markers}")
plt.show()

## 4. Comprehensive Phenotyping Validation Analysis

This step performs a comprehensive validation of all phenotyping methods against a designated ground truth column (`phenotype_ref1`). It evaluates individual methods, calculates a consensus phenotype, and generates a series of detailed reports and visualizations.

The core logic is encapsulated in the `src/comparison_utils.py` module. The following cell configures the analysis and executes the main `run_analysis` function.

In [ ]:
from pathlib import Path
import sys
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
# Use the script's location to define a robust project root path
project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from spacemap.comparison_utils import run_analysis

INPUT_FILE = project_root / 'outputs/phenotyping' / 'all_samples_combined_corrected_phenotyped.csv'
#INPUT_FILE = project_root / 'outputs/phenotyping' / 'all_samples_combined_corrected_phenotyped_ml_mr.csv'

OUTPUT_DIR = project_root / 'outputs/phenotyping/comparison/ML_all'

GROUND_TRUTH_COLUMN = 'phenotype_ref2'
DATASET_ID_COLUMN = 'batch_id'

# Phenotyping methods to analyze
PHENOTYPING_METHODS = [
    'phenotype_leiden',
    'phenotype_som',
    'phenotype_threshold',
    'phenotype_scimap',
    'phenotype_ml_mr',
    'phenotype_ml_consensus',
    'phenotype_consensus_refined'
]

# Consensus requirements
MIN_CONSENSUS_METHODS = 3
PERFORM_CONSENSUS_ANALYSIS = True  # Set to False to skip consensus analysis

# Figure settings
FIGURE_DPI = 300

# =============================================================================
# EXECUTION
# =============================================================================
if __name__ == "__main__":
    run_analysis(
        input_file=INPUT_FILE,
        output_dir=OUTPUT_DIR,
        ground_truth_column=GROUND_TRUTH_COLUMN,
        dataset_id_column=DATASET_ID_COLUMN,
        phenotyping_methods=PHENOTYPING_METHODS,
        min_consensus_methods=MIN_CONSENSUS_METHODS,
        figure_dpi=FIGURE_DPI,
        perform_consensus_analysis=PERFORM_CONSENSUS_ANALYSIS,
    )


In [ ]:
# Stratification based on ml_consensus_score

from pathlib import Path
import sys
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
# Use the script's location to define a robust project root path
project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from spacemap.comparison_utils import run_analysis

INPUT_FILE = project_root / 'outputs/phenotyping' / 'all_samples_combined_corrected_phenotyped.csv'
#INPUT_FILE = project_root / 'outputs/phenotyping' / 'all_samples_combined_corrected_phenotyped_ml_mr.csv'

OUTPUT_DIR = project_root / 'outputs/phenotyping/comparison/ML_consensus_stratified'

GROUND_TRUTH_COLUMN = 'phenotype_ref1'
DATASET_ID_COLUMN = 'file_prefix'

# =============================================================================
# DATA PREPROCESSING - Add stratified columns based on ml_consensus_score
# =============================================================================
print("Loading data and creating stratified phenotype columns...")

# Load the data to add stratified columns
df = pd.read_csv(INPUT_FILE)

# Define score thresholds for stratification
score_thresholds = {
    'phenotype_ml_consensus_0': 0.0,
    'phenotype_ml_consensus_20': 0.20,
    'phenotype_ml_consensus_40': 0.40,
    'phenotype_ml_consensus_60': 0.60,
    'phenotype_ml_consensus_80': 0.80
}

# Create stratified phenotype columns
for new_col, threshold in score_thresholds.items():
    if 'ml_consensus_score' in df.columns:
        # Create a copy of ml_consensus column, but set to NaN where score < threshold
        df[new_col] = df['phenotype_ml_consensus'].where(
            df['ml_consensus_score'] >= threshold, 
            other=pd.NA
        )
        valid_count = df[new_col].notna().sum()
        total_count = len(df)
        print(f"Created {new_col}: {valid_count}/{total_count} cells meet threshold >= {threshold} ({valid_count/total_count*100:.1f}%)")
    else:
        print(f"Warning: 'ml_consensus_score' column not found. Cannot create {new_col}")
        df[new_col] = pd.NA

# Save the modified dataframe temporarily for analysis
temp_input_file = OUTPUT_DIR / 'temp_stratified_data.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(temp_input_file, index=False)
print(f"Saved stratified data to: {temp_input_file}")

# Phenotyping methods to analyze (now includes stratified versions)
PHENOTYPING_METHODS = [
    'phenotype_ml_mr',
    'phenotype_ml_consensus',
    'phenotype_ml_consensus_0',
    'phenotype_ml_consensus_20',
    'phenotype_ml_consensus_40',
    'phenotype_ml_consensus_60',
    'phenotype_ml_consensus_80'
]

# Consensus requirements
MIN_CONSENSUS_METHODS = 3
PERFORM_CONSENSUS_ANALYSIS = True  # Set to False to skip consensus analysis

# Figure settings
FIGURE_DPI = 300

# =============================================================================
# EXECUTION
# =============================================================================
if __name__ == "__main__":
    print(f"\nRunning analysis with {len(PHENOTYPING_METHODS)} phenotyping methods:")
    for method in PHENOTYPING_METHODS:
        print(f"  - {method}")
    
    run_analysis(
        input_file=temp_input_file,  # Use the modified file with stratified columns
        output_dir=OUTPUT_DIR,
        ground_truth_column=GROUND_TRUTH_COLUMN,
        dataset_id_column=DATASET_ID_COLUMN,
        phenotyping_methods=PHENOTYPING_METHODS,
        min_consensus_methods=MIN_CONSENSUS_METHODS,
        figure_dpi=FIGURE_DPI,
        perform_consensus_analysis=PERFORM_CONSENSUS_ANALYSIS,
    )
    
    # Clean up temporary file
    if temp_input_file.exists():
        temp_input_file.unlink()
        print(f"\nCleaned up temporary file: {temp_input_file}")
    
    print(f"\nAnalysis complete! Results saved to: {OUTPUT_DIR}")
    print("The F1 score plot and other figures now include stratified ml_consensus columns based on confidence scores.")


### This cell performs batch phenotype method comparison across multiple CSV files. Here's what it does:

Purpose:
Analyzes agreement between different phenotyping methods across all CSV files in a directory and generates a summary comparison report.

In [ ]:
import pandas as pd
import os, sys
import glob
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
# Use the script's location to define a robust project root path
project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    

def analyze_phenotype_files():
    """
    Scans CSV files in a specified directory, performs pairwise comparisons
    on phenotype columns, and saves the matching percentages to a new CSV file.
    """
    # 1. Define the directory and columns for comparison
    # --- IMPORTANT: Update this path if your folder is located elsewhere ---
    directory_path = project_root / 'outputs/phenotyping/batch/circles_v2'

    methods_to_compare = [
        'phenotype_leiden',
        'phenotype_som',
        'phenotype_threshold',
        'phenotype_scimap',
        'phenotype_ml_mr',
        'phenotype_ml_consensus',
        'phenotype_consensus_refined'

    ]
    
    # Create a mapping for shorter column names in the output file
    name_map = {
        'phenotype_leiden': 'leiden',
        'phenotype_som': 'som',
        'phenotype_threshold': 'threshold',
        'phenotype_scimap': 'scimap',
        'phenotype_ml_mr': 'ml_mr',
        'phenotype_ml_consensus': 'ml_consensus',
        'phenotype_consensus_refined': 'ml_consensus_refined'

    }

    # 2. Find all CSV files in the target directory
    csv_files = glob.glob(os.path.join(directory_path, '*.csv'))
    
    # Exclude the potential output file if the script is run multiple times
    output_filename = 'pairwise_comparision_circles.csv'
    csv_files = [f for f in csv_files if not f.endswith(output_filename)]

    if not csv_files:
        print(f"No CSV files found in the directory: {directory_path}")
        return

    print(f"Found {len(csv_files)} CSV files to process.")

    # 3. Process each file and store results
    all_results = []
    
    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path)
            file_name = os.path.basename(file_path)
            
            # Check if all required columns exist in the current file
            if not all(col in df.columns for col in methods_to_compare):
                print(f"Skipping {file_name}: missing one or more required columns.")
                continue

            # Dictionary to store results for the current file
            result_row = {'file': file_name}
            
            # 4. Perform pairwise comparison
            # Generate all unique pairs of columns, e.g., (leiden, som), (leiden, threshold), etc.
            for col1, col2 in combinations(methods_to_compare, 2):
                
                # Calculate the percentage of matching values
                matches = (df[col1] == df[col2]).sum()
                total_rows = len(df)
                
                # Avoid division by zero if the dataframe is empty
                match_percentage = (matches / total_rows) * 100 if total_rows > 0 else 0
                
                # Create a clean column name for the output file (e.g., 'leiden_som')
                output_col_name = f"{name_map[col1]}_{name_map[col2]}"
                
                # Store the result
                result_row[output_col_name] = match_percentage
            
            all_results.append(result_row)
            print(f"Processed: {file_name}")

        except Exception as e:
            print(f"Could not process file {os.path.basename(file_path)}. Error: {e}")

    # 5. Save the combined results to a new CSV file
    if not all_results:
        print("No data was processed. Output file will not be created.")
        return
        
    output_df = pd.DataFrame(all_results)
    output_path = os.path.join(directory_path, output_filename)
    
    # Reorder columns to have the 'file' column first
    cols = ['file'] + [col for col in output_df.columns if col != 'file']
    output_df = output_df[cols]
    
    output_df.to_csv(output_path, index=False)
    
    print(f"\n Success! Results saved to: {output_path}")


# Run the analysis
if __name__ == '__main__':
    analyze_phenotype_files()